<a href="https://colab.research.google.com/github/sumaiiiyyah/code/blob/master/31_Ligand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZINC-Priority 31 Natural-Product Ligand Library — PubChem-Only Automated Workflow

**Purpose:** use the 31 fixed parent compounds and their supplied PubChem CIDs to retrieve parent structures, retrieve PubChem 2D-similar compounds, apply RDKit QC, generate and validate 3D SDF libraries, create audit tables and package the run as a ZIP.

## Important provenance boundary

- CartBlanche/ZINC-22 is skipped completely. This notebook makes **no request** to `cartblanche22.docking.org` and requires **no manual ZINC export or upload**.
- The supplied ZINC IDs remain unverified cross-references from the selection dossier. They are retained as metadata only; this workflow does not establish current ZINC-22 record existence, supplier/catalogue status or purchasability.
- Parent structures are retrieved by the supplied PubChem CIDs. The notebook checks that each CID returns a valid structure and records PubChem identifiers. A CID lookup alone does not independently prove the dossier's name↔CID mapping or ZINC↔PubChem identity.
- All reported counts are generated by the current run. No historical 24-parent results are reused.
- No protein target, docking, protein preparation or binding-site analysis is assumed.


## 1. Install and import packages

In [1]:
!pip -q install rdkit pubchempy pandas requests python-docx

import os, re, io, json, time, zipfile, shutil, textwrap, hashlib, traceback
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote
import requests
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.inchi import MolToInchiKey
from IPython.display import display
from google.colab import files

print("RDKit:", __import__("rdkit").__version__)
print("pandas:", pd.__version__)
print("Environment ready.")

RDKit: 2026.09.1
pandas: 2.2.3
Environment ready.


## 2. Define the 31-parent panel and provenance metadata

The panel is fixed from the supplied 31-parent list. ZINC IDs are retained as **unverified cross-reference metadata**. PubChem CIDs are the structure-retrieval keys for this no-CartBlanche workflow.


In [2]:
PANEL = [
    {"name":"Resveratrol","zinc_id":"ZINC000000006787","pubchem_cid":445154,"chembl_id":"CHEMBL165","origin":"Vitis vinifera; Reynoutria japonica (syn. Polygonum cuspidatum); other plants","pubmed_start":"32449796;36499286"},
    {"name":"Genistin","zinc_id":"ZINC000004097913","pubchem_cid":5281377,"chembl_id":"CHEMBL486625","origin":"Glycine max (soybean) and other legumes","pubmed_start":"31192770"},
    {"name":"Luteolin","zinc_id":"ZINC000018185774","pubchem_cid":5280445,"chembl_id":"CHEMBL151","origin":"Parsley, celery, chamomile and numerous plants","pubmed_start":"36499286;31192770"},
    {"name":"Plumbagin","zinc_id":"ZINC000000058187","pubchem_cid":10205,"chembl_id":"CHEMBL295316","origin":"Plumbago zeylanica and other Plumbaginaceae, Ebenaceae and Droseraceae plants","pubmed_start":"28359198;33344645"},
    {"name":"Withaferin A","zinc_id":"ZINC000008234189","pubchem_cid":265237,"chembl_id":"CHEMBL517080","origin":"Withania somnifera (ashwagandha), particularly leaves; other Withania species","pubmed_start":"32727824;37259311;33434782"},
    {"name":"Berberine","zinc_id":"ZINC000003779067","pubchem_cid":2353,"chembl_id":"CHEMBL295124","origin":"Berberis spp., Coptis chinensis and other Berberidaceae","pubmed_start":"32449796;21777476"},
    {"name":"Celastrol","zinc_id":"ZINC000019795938","pubchem_cid":122724,"chembl_id":"CHEMBL301982","origin":"Tripterygium wilfordii roots and other Celastraceae plant sources","pubmed_start":"33364939;37196541"},
    {"name":"Genistein","zinc_id":"ZINC000018825330","pubchem_cid":5280961,"chembl_id":"CHEMBL44","origin":"Glycine max and other legumes","pubmed_start":"31192770;36499286"},
    {"name":"Oleanolic acid","zinc_id":"ZINC000003785416","pubchem_cid":10494,"chembl_id":"CHEMBL168","origin":"Olea europaea, Ligustrum spp. and many plants","pubmed_start":"24486850;34798468"},
    {"name":"Betulinic acid","zinc_id":"ZINC000004097714","pubchem_cid":64971,"chembl_id":"CHEMBL269277","origin":"Betula spp. bark and other plant sources","pubmed_start":"34269920"},
    {"name":"Kaempferol","zinc_id":"ZINC000003869768","pubchem_cid":5280863,"chembl_id":"CHEMBL150","origin":"Tea, broccoli, kale, beans, apples and other plants","pubmed_start":"31192770"},
    {"name":"Ursolic acid","zinc_id":"ZINC000003978827","pubchem_cid":64945,"chembl_id":"CHEMBL169","origin":"Apple peel, rosemary, holy basil and other plants","pubmed_start":"22149093;34269920"},
    {"name":"Apigenin","zinc_id":"ZINC000003871576","pubchem_cid":5280443,"chembl_id":"CHEMBL28","origin":"Parsley, celery, chamomile and other plants","pubmed_start":"31192770"},
    {"name":"Piperine","zinc_id":"ZINC000001529772","pubchem_cid":638024,"chembl_id":"CHEMBL43185","origin":"Piper nigrum and Piper longum fruits","pubmed_start":"22149093"},
    {"name":"Piperlongumine","zinc_id":"ZINC000000899053","pubchem_cid":637858,"chembl_id":"CHEMBL465843","origin":"Piper longum and related Piper species","pubmed_start":"34503074"},
    {"name":"Baicalein","zinc_id":"ZINC000003871633","pubchem_cid":5281605,"chembl_id":"CHEMBL8260","origin":"Scutellaria baicalensis root and related Scutellaria spp.","pubmed_start":"28008217"},
    {"name":"Shikonin","zinc_id":"ZINC000002015152","pubchem_cid":479503,"chembl_id":"CHEMBL9470","origin":"Lithospermum erythrorhizon and Arnebia spp.","pubmed_start":"36752292;35847041;30858045"},
    {"name":"Magnolol","zinc_id":"ZINC000000001645","pubchem_cid":72300,"chembl_id":"CHEMBL180920","origin":"Magnolia officinalis and other Magnolia spp.","pubmed_start":"38714288"},
    {"name":"Honokiol","zinc_id":"ZINC000000001536","pubchem_cid":72303,"chembl_id":"CHEMBL16901","origin":"Magnolia officinalis and other Magnolia spp.","pubmed_start":"38714288"},
    {"name":"Wogonin","zinc_id":"ZINC000000899093","pubchem_cid":5281703,"chembl_id":"CHEMBL16171","origin":"Scutellaria baicalensis and related Scutellaria spp.","pubmed_start":"21777476;31192770"},
    {"name":"Harmine","zinc_id":"ZINC000018847046","pubchem_cid":5280953,"chembl_id":"CHEMBL269538","origin":"Peganum harmala and other harmala-containing plants","pubmed_start":"36656035"},
    {"name":"Artemisinin","zinc_id":"ZINC000008143788","pubchem_cid":68827,"chembl_id":"CHEMBL269671","origin":"Artemisia annua","pubmed_start":"39202965;21777476"},
    {"name":"Thymoquinone","zinc_id":"ZINC000000164367","pubchem_cid":10281,"chembl_id":"CHEMBL1672002","origin":"Nigella sativa seeds and seed oil","pubmed_start":"20924969;22149093"},
    {"name":"Andrographolide","zinc_id":"ZINC000003881797","pubchem_cid":5318517,"chembl_id":"CHEMBL186141","origin":"Andrographis paniculata aerial parts","pubmed_start":"38255306;34269920"},
    {"name":"Diosgenin","zinc_id":"ZINC000006857614","pubchem_cid":99474,"chembl_id":"CHEMBL412437","origin":"Dioscorea spp. and Trigonella foenum-graecum","pubmed_start":"22149093"},
    {"name":"Capsaicin","zinc_id":"ZINC000001530575","pubchem_cid":1548943,"chembl_id":"CHEMBL294199","origin":"Capsicum spp. fruits","pubmed_start":"22149093"},
    {"name":"Eugenol","zinc_id":"ZINC000000001411","pubchem_cid":3314,"chembl_id":"CHEMBL42710","origin":"Clove (Syzygium aromaticum) and other aromatic plants","pubmed_start":"34885992;36152592"},
    {"name":"Naringenin","zinc_id":"ZINC000000156701","pubchem_cid":439246,"chembl_id":"CHEMBL9352","origin":"Citrus fruits and other plants","pubmed_start":"31192770"},
    {"name":"Ellagic acid","zinc_id":"ZINC000003872446","pubchem_cid":5281855,"chembl_id":"CHEMBL6246","origin":"Pomegranate, raspberries, strawberries, walnuts and other plants","pubmed_start":"38255306"},
    {"name":"Caffeic acid","zinc_id":"ZINC000000058172","pubchem_cid":689043,"chembl_id":"CHEMBL145","origin":"Coffee, fruits, vegetables, herbs and other plants","pubmed_start":"36499286"},
    {"name":"Myricetin","zinc_id":"ZINC000003874317","pubchem_cid":5281672,"chembl_id":"CHEMBL164","origin":"Berries, grapes, tea and other plants","pubmed_start":"31192770"},
]
assert len(PANEL) == 31, f"Expected 31 parents, found {len(PANEL)}"
assert len({x["zinc_id"] for x in PANEL}) == 31, "Duplicate ZINC IDs detected"
assert len({x["pubchem_cid"] for x in PANEL}) == 31, "Duplicate PubChem CIDs detected"
panel_df = pd.DataFrame(PANEL)
display(panel_df)

,name,zinc_id,pubchem_cid,chembl_id,origin,pubmed_start
0,Resveratrol,ZINC000000006787,445154,CHEMBL165,Vitis vinifera; Reynoutria japonica (syn. Poly...,32449796;36499286
1,Genistin,ZINC000004097913,5281377,CHEMBL486625,Glycine max (soybean) and other legumes,31192770
2,Luteolin,ZINC000018185774,5280445,CHEMBL151,"Parsley, celery, chamomile and numerous plants",36499286;31192770
3,Plumbagin,ZINC000000058187,10205,CHEMBL295316,"Plumbago zeylanica and other Plumbaginaceae, E...",28359198;33344645
4,Withaferin A,ZINC000008234189,265237,CHEMBL517080,"Withania somnifera (ashwagandha), particularly...",32727824;37259311;33434782
5,Berberine,ZINC000003779067,2353,CHEMBL295124,"Berberis spp., Coptis chinensis and other Berb...",32449796;21777476
6,Celastrol,ZINC000019795938,122724,CHEMBL301982,Tripterygium wilfordii roots and other Celastr...,33364939;37196541
7,Genistein,ZINC000018825330,5280961,CHEMBL44,Glycine max and other legumes,31192770;36499286
8,Oleanolic acid,ZINC000003785416,10494,CHEMBL168,"Olea europaea, Ligustrum spp. and many plants",24486850;34798468
9,Betulinic acid,ZINC000004097714,64971,CHEMBL269277,Betula spp. bark and other plant sources,34269920


## 3. Configure paths and fixed workflow parameters

In [3]:
# Use a unique run directory so rerunning the notebook never deletes a previous run.
BASE_PROJECT_DIR = Path("/content/ZINC_Priority_31_Ligand_Library_FINAL")
if BASE_PROJECT_DIR.exists():
    RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    PROJECT_DIR = BASE_PROJECT_DIR.with_name(f"{BASE_PROJECT_DIR.name}_{RUN_STAMP}")
else:
    PROJECT_DIR = BASE_PROJECT_DIR

LIGAND_ROOT = PROJECT_DIR / "01_Ligands"
PROTEIN_ROOT = PROJECT_DIR / "02_Proteins"
SUMMARY_ROOT = PROJECT_DIR / "03_Summary"

for p in [LIGAND_ROOT, PROTEIN_ROOT, SUMMARY_ROOT]:
    p.mkdir(parents=True, exist_ok=False)

SIMILARITY_THRESHOLD = 90
MAX_RECORDS_PER_PARENT = 1000
REQUEST_TIMEOUT = (10, 120)
MAX_RETRIES = 3
RANDOM_SEED = 42
PERMITTED_ELEMENTS = {"C","H","N","O","S","P","F","Cl","Br","I"}
METAL_ELEMENTS = set("Li Na K Rb Cs Be Mg Ca Sr Ba Al Ga In Tl Sc Y La Ti Zr Hf V Nb Ta Cr Mo W Mn Tc Re Fe Ru Os Co Rh Ir Ni Pd Pt Cu Ag Au Zn Cd Hg Gd Tb Dy Ho Er Tm Yb Lu U Th".split())

for i, row in enumerate(PANEL, start=1):
    folder = LIGAND_ROOT / f"{i:02d}_{re.sub(r'[^A-Za-z0-9]+','_',row['name']).strip('_')}"
    (folder/"raw").mkdir(parents=True, exist_ok=False)
    (folder/"QC_clean").mkdir(parents=True, exist_ok=False)
    row["folder"] = folder

print("Project directory:", PROJECT_DIR)
print("Parent libraries:", len(PANEL))
print("Similarity:", SIMILARITY_THRESHOLD, "% 2D Tanimoto")
print("Maximum records per parent:", MAX_RECORDS_PER_PARENT)
print("Existing project data will not be deleted.")

pd.DataFrame([{k:v for k,v in row.items() if k != "folder"} for row in PANEL]).to_csv(
    SUMMARY_ROOT/"31_Parent_Input_Manifest.csv", index=False
)


Project directory: /content/ZINC_Priority_31_Ligand_Library_FINAL
Parent libraries: 31
Similarity: 90 % 2D Tanimoto
Maximum records per parent: 1000
Existing project data will not be deleted.


## 4. General API request helpers

In [4]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent":"ZINC-Priority-Ligand-Library/2.0 (academic research; contact: user)"})

def request_with_retries(method, url, **kwargs):
    last_error = None
    kwargs.setdefault("timeout", REQUEST_TIMEOUT)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = SESSION.request(method, url, **kwargs)
            if response.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Temporary HTTP {response.status_code}: {response.text[:300]}")
            response.raise_for_status()
            return response
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                wait = 2 ** attempt
                print(f"Retry {attempt}/{MAX_RETRIES-1} in {wait}s: {exc}")
                time.sleep(wait)
    raise RuntimeError(f"Request failed after {MAX_RETRIES} attempts: {last_error}")

def parse_tabular_text(text):
    text = text.strip()
    if not text:
        return pd.DataFrame()
    sep = "\t" if "\t" in text.splitlines()[0] else ","
    return pd.read_csv(io.StringIO(text), sep=sep)

print("Request helpers ready.")


Request helpers ready.


## 5. PubChem-only parent structure retrieval — ZINC skipped

This cell intentionally replaces the former ZINC browser-export upload step. It performs no CartBlanche request and asks for no file upload. It creates a provenance table marking every ZINC identifier as unverified, then Cell 6 retrieves the parent structures directly from PubChem using the supplied CIDs. If PubChem is unavailable from this runtime, processing stops with a clear PubChem network/API error; no structure is fabricated.


In [5]:
# ZINC is intentionally skipped. No CartBlanche request and no manual upload.
zinc_ids = [x["zinc_id"] for x in PANEL]
zinc_provenance_df = pd.DataFrame([{
    "Compound": x["name"],
    "ZINC_ID_Cross_Reference": x["zinc_id"],
    "ZINC22_Live_Status": "NOT_QUERIED_SKIPPED_AS_REQUESTED",
    "PubChem_CID_Supplied": x["pubchem_cid"],
} for x in PANEL])
zinc_provenance_df.to_csv(SUMMARY_ROOT / "ZINC_Cross_Reference_Not_Live_Verified.csv", index=False)
print("CartBlanche skipped. No ZINC request was made and no upload is required.")
print(f"Fixed parent compounds: {len(PANEL)}")
print("ZINC IDs are metadata only and will not be described as live-verified.")
display(zinc_provenance_df)


CartBlanche skipped. No ZINC request was made and no upload is required.
Fixed parent compounds: 31
ZINC IDs are metadata only and will not be described as live-verified.


,Compound,ZINC_ID_Cross_Reference,ZINC22_Live_Status,PubChem_CID_Supplied
0,Resveratrol,ZINC000000006787,NOT_QUERIED_SKIPPED_AS_REQUESTED,445154
1,Genistin,ZINC000004097913,NOT_QUERIED_SKIPPED_AS_REQUESTED,5281377
2,Luteolin,ZINC000018185774,NOT_QUERIED_SKIPPED_AS_REQUESTED,5280445
3,Plumbagin,ZINC000000058187,NOT_QUERIED_SKIPPED_AS_REQUESTED,10205
4,Withaferin A,ZINC000008234189,NOT_QUERIED_SKIPPED_AS_REQUESTED,265237
5,Berberine,ZINC000003779067,NOT_QUERIED_SKIPPED_AS_REQUESTED,2353
6,Celastrol,ZINC000019795938,NOT_QUERIED_SKIPPED_AS_REQUESTED,122724
7,Genistein,ZINC000018825330,NOT_QUERIED_SKIPPED_AS_REQUESTED,5280961
8,Oleanolic acid,ZINC000003785416,NOT_QUERIED_SKIPPED_AS_REQUESTED,10494
9,Betulinic acid,ZINC000004097714,NOT_QUERIED_SKIPPED_AS_REQUESTED,64971


## 6. Retrieve PubChem parent identity records

Parent structures are fetched by the supplied PubChem CIDs using documented PUG-REST property names (`SMILES` and `ConnectivitySMILES`). This confirms that a CID resolves to a parseable PubChem structure; it does not validate the ZINC cross-reference. Any missing CID, missing isomeric/canonical SMILES or invalid structure stops the workflow before similarity retrieval.


In [6]:
# PubChem PUG-REST properties: use only documented property tags.
# PubChem's stereochemistry-bearing field is SMILES; ConnectivitySMILES omits stereo.
PUBCHEM = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
PUBCHEM_PROPERTY_FIELDS = (
    "Title,IUPACName,MolecularFormula,MolecularWeight,"
    "SMILES,ConnectivitySMILES,InChI,InChIKey"
)

def normalize_pubchem_record(record):
    """Normalize official PUG-REST property names for downstream notebook cells."""
    rec = dict(record)
    if rec.get("SMILES"):
        rec["IsomericSMILES"] = rec["SMILES"]
    if rec.get("ConnectivitySMILES"):
        rec["CanonicalSMILES"] = rec["ConnectivitySMILES"]
    return rec

def pubchem_parent(cid):
    url = f"{PUBCHEM}/compound/cid/{cid}/property/{PUBCHEM_PROPERTY_FIELDS}/JSON"
    r = request_with_retries("GET", url)
    js = r.json()
    records = js.get("PropertyTable", {}).get("Properties", [])
    if len(records) != 1:
        raise RuntimeError(
            f"PubChem CID {cid}: expected one property record, received {len(records)}; "
            f"response={str(js)[:500]}"
        )
    return normalize_pubchem_record(records[0])

def mol_from_smiles(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    return Chem.MolFromSmiles(smiles)

def canonical_isomeric(smiles):
    m = mol_from_smiles(smiles)
    if m is None:
        return None
    return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)

def inchikey_from_smiles(smiles):
    m = mol_from_smiles(smiles)
    if m is None:
        return None
    try:
        return MolToInchiKey(m)
    except Exception:
        return None

parent_audit = []
parent_props = {}
for row in PANEL:
    name, zid, cid = row["name"], row["zinc_id"], int(row["pubchem_cid"])
    p = {}
    try:
        p = pubchem_parent(cid)
        psmiles = p.get("IsomericSMILES") or p.get("CanonicalSMILES")
        pmol = mol_from_smiles(psmiles)
        if pmol is None:
            raise RuntimeError("PubChem returned missing or invalid SMILES")
        pkey = p.get("InChIKey") or inchikey_from_smiles(psmiles)
        if not pkey:
            raise RuntimeError("PubChem structure did not yield an InChIKey")
        parent_props[cid] = p
        status = "PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED"
        error = ""
    except Exception as exc:
        parent_props[cid] = {"lookup_error": str(exc)}
        psmiles = pkey = None
        status = "PUBCHEM_LOOKUP_FAILED"
        error = str(exc)
    parent_audit.append({
        "Compound": name,
        "ZINC_ID_Cross_Reference": zid,
        "ZINC22_Live_Status": "NOT_QUERIED_SKIPPED_AS_REQUESTED",
        "PubChem_CID": cid,
        "PubChem_Title": p.get("Title", "") if status != "PUBCHEM_LOOKUP_FAILED" else "",
        "PubChem_Molecular_Formula": p.get("MolecularFormula", "") if status != "PUBCHEM_LOOKUP_FAILED" else "",
        "PubChem_InChIKey": pkey,
        "PubChem_Isomeric_SMILES": psmiles,
        "PubChem_Status": status,
        "Failure_Reason": error,
    })
    time.sleep(0.25)

parent_audit_df = pd.DataFrame(parent_audit)
parent_audit_df.to_csv(SUMMARY_ROOT / "PubChem_Parent_CID_Structure_Audit.csv", index=False)
display(parent_audit_df[["Compound", "PubChem_CID", "PubChem_Title", "PubChem_Status", "Failure_Reason"]])

passed = parent_audit_df[parent_audit_df["PubChem_Status"] == "PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED"]
failed = parent_audit_df[parent_audit_df["PubChem_Status"] != "PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED"]
print(f"PubChem parent structures retrieved: {len(passed)}/31")
print(f"PubChem parent structures unresolved: {len(failed)}/31")
if len(passed) != len(PANEL):
    failed.to_csv(SUMMARY_ROOT / "PubChem_Parent_CID_Failures.csv", index=False)
    first_reason = failed.iloc[0]["Failure_Reason"] if len(failed) else "Unknown error"
    raise RuntimeError(
        f"STOP: only {len(passed)}/31 supplied PubChem CIDs returned valid structures. "
        f"First failure: {first_reason}. "
        "No similarity library or SDF package was completed."
    )


,Compound,PubChem_CID,PubChem_Title,PubChem_Status,Failure_Reason
0,Resveratrol,445154,Resveratrol,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
1,Genistin,5281377,Genistin,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
2,Luteolin,5280445,Luteolin,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
3,Plumbagin,10205,Plumbagin,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
4,Withaferin A,265237,Withaferin A,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
5,Berberine,2353,Berberine,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
6,Celastrol,122724,Celastrol,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
7,Genistein,5280961,Genistein,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
8,Oleanolic acid,10494,Oleanolic Acid,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,
9,Betulinic acid,64971,Betulinic Acid,PUBCHEM_CID_STRUCTURE_RETRIEVED_ZINC_UNVERIFIED,


PubChem parent structures retrieved: 31/31
PubChem parent structures unresolved: 0/31


## 7. PubChem fast 2D similarity retrieval

For each validated parent PubChem CID, retrieve up to 1,000 CIDs at a 90% 2D Tanimoto threshold. The parent CID is explicitly retained. Duplicate CIDs are removed while preserving order. Raw CID lists are preserved before property retrieval.

In [7]:
def pubchem_similarity_cids(cid, threshold=SIMILARITY_THRESHOLD, max_records=MAX_RECORDS_PER_PARENT):
    url = f"{PUBCHEM}/compound/fastsimilarity_2d/cid/{cid}/cids/JSON"
    params = {"Threshold":threshold, "MaxRecords":max_records}
    r = request_with_retries("GET", url, params=params)
    try:
        js = r.json()
    except Exception as exc:
        raise RuntimeError(f"PubChem similarity endpoint returned non-JSON content: {r.text[:300]}") from exc
    cids = js.get("IdentifierList", {}).get("CID")
    if cids is None:
        # A 202/Waiting response or ListKey response is not a completed hit list.
        raise RuntimeError(f"PubChem did not return a completed CID list: {str(js)[:500]}")
    return [int(x) for x in cids]

retrieval_summary = []
raw_cid_lists = {}
for i, row in enumerate(PANEL, start=1):
    cid = int(row["pubchem_cid"])
    try:
        hits = pubchem_similarity_cids(cid)
        # Parent retention + ordered de-duplication
        ordered = list(dict.fromkeys(hits + [cid]))
        raw_cid_lists[cid] = ordered
        raw_dir = row["folder"]/"raw"
        pd.DataFrame({"PubChem_CID":ordered}).to_csv(raw_dir/f"{row['name']}_raw_similarity_CIDs.csv", index=False)
        retrieval_summary.append({
            "Parent_Ligand":row["name"],"Parent_CID":cid,
            "Similarity_Threshold_Percent":SIMILARITY_THRESHOLD,
            "MaxRecords":MAX_RECORDS_PER_PARENT,
            "PubChem_Hits_Returned":len(hits),
            "Parent_Retained":cid in ordered,
            "Unique_Candidate_CIDs":len(ordered),
            "Retrieval_Status":"SUCCESS"
        })
    except Exception as exc:
        raw_cid_lists[cid] = []
        retrieval_summary.append({
            "Parent_Ligand":row["name"],"Parent_CID":cid,
            "Similarity_Threshold_Percent":SIMILARITY_THRESHOLD,
            "MaxRecords":MAX_RECORDS_PER_PARENT,
            "PubChem_Hits_Returned":0,"Parent_Retained":False,
            "Unique_Candidate_CIDs":0,"Retrieval_Status":f"FAILED: {exc}"
        })
    print(f"{i:02d}/31 {row['name']}: {len(raw_cid_lists[cid])} unique CIDs")
    time.sleep(0.5)

retrieval_df = pd.DataFrame(retrieval_summary)
retrieval_df.to_csv(SUMMARY_ROOT/"PubChem_Similarity_Retrieval_Summary.csv", index=False)
display(retrieval_df)

01/31 Resveratrol: 1000 unique CIDs
02/31 Genistin: 1000 unique CIDs
03/31 Luteolin: 1000 unique CIDs
04/31 Plumbagin: 1000 unique CIDs
05/31 Withaferin A: 970 unique CIDs
06/31 Berberine: 1000 unique CIDs
07/31 Celastrol: 454 unique CIDs
08/31 Genistein: 1000 unique CIDs
09/31 Oleanolic acid: 1000 unique CIDs
10/31 Betulinic acid: 1000 unique CIDs
11/31 Kaempferol: 1000 unique CIDs
12/31 Ursolic acid: 1000 unique CIDs
13/31 Apigenin: 1000 unique CIDs
14/31 Piperine: 1000 unique CIDs
15/31 Piperlongumine: 947 unique CIDs
16/31 Baicalein: 1000 unique CIDs
17/31 Shikonin: 757 unique CIDs
18/31 Magnolol: 1000 unique CIDs
19/31 Honokiol: 1000 unique CIDs
20/31 Wogonin: 1000 unique CIDs
21/31 Harmine: 1000 unique CIDs
22/31 Artemisinin: 662 unique CIDs
23/31 Thymoquinone: 207 unique CIDs
24/31 Andrographolide: 597 unique CIDs
25/31 Diosgenin: 805 unique CIDs
26/31 Capsaicin: 1000 unique CIDs
27/31 Eugenol: 1000 unique CIDs
28/31 Naringenin: 1000 unique CIDs
29/31 Ellagic acid: 1000 unique C

,Parent_Ligand,Parent_CID,Similarity_Threshold_Percent,MaxRecords,PubChem_Hits_Returned,Parent_Retained,Unique_Candidate_CIDs,Retrieval_Status
0,Resveratrol,445154,90,1000,1000,True,1000,SUCCESS
1,Genistin,5281377,90,1000,1000,True,1000,SUCCESS
2,Luteolin,5280445,90,1000,1000,True,1000,SUCCESS
3,Plumbagin,10205,90,1000,1000,True,1000,SUCCESS
4,Withaferin A,265237,90,1000,970,True,970,SUCCESS
5,Berberine,2353,90,1000,1000,True,1000,SUCCESS
6,Celastrol,122724,90,1000,454,True,454,SUCCESS
7,Genistein,5280961,90,1000,1000,True,1000,SUCCESS
8,Oleanolic acid,10494,90,1000,1000,True,1000,SUCCESS
9,Betulinic acid,64971,90,1000,1000,True,1000,SUCCESS


## 8. Retrieve PubChem chemical properties in batches and preserve raw metadata

In [8]:
# Batch property retrieval using documented PubChem PUG-REST property tags.
PROPERTY_FIELDS = PUBCHEM_PROPERTY_FIELDS

def pubchem_properties_batch(cids):
    cid_string = ",".join(str(int(x)) for x in cids)
    url = f"{PUBCHEM}/compound/cid/{cid_string}/property/{PROPERTY_FIELDS}/JSON"
    r = request_with_retries("GET", url)
    payload = r.json()
    records = payload.get("PropertyTable", {}).get("Properties", [])
    return [normalize_pubchem_record(x) for x in records]

candidate_records = {}
for row in PANEL:
    parent_cid = int(row["pubchem_cid"])
    cids = raw_cid_lists[parent_cid]
    records = []
    for start in range(0, len(cids), 100):
        batch = cids[start:start+100]
        try:
            batch_records = pubchem_properties_batch(batch)
            returned = {int(x["CID"]) for x in batch_records if x.get("CID") is not None}
            missing = sorted(set(map(int, batch)) - returned)
            if missing:
                raise RuntimeError(f"PubChem omitted {len(missing)} requested CID(s): {missing[:10]}")
            records.extend(batch_records)
        except Exception as exc:
            raise RuntimeError(
                f"STOP: property retrieval failed for {row['name']} batch starting at {start}: {exc}"
            ) from exc
        time.sleep(0.5)
    candidate_records[parent_cid] = records
    raw_df = pd.DataFrame(records)
    raw_df.to_csv(row["folder"]/"raw"/f"{row['name']}_raw_candidates.csv", index=False)
    print(row["name"], "properties retrieved:", len(records))

print("Property retrieval stage completed.")


Resveratrol properties retrieved: 1000
Genistin properties retrieved: 1000
Luteolin properties retrieved: 1000
Plumbagin properties retrieved: 1000
Withaferin A properties retrieved: 970
Berberine properties retrieved: 1000
Celastrol properties retrieved: 454
Genistein properties retrieved: 1000
Oleanolic acid properties retrieved: 1000
Betulinic acid properties retrieved: 1000
Kaempferol properties retrieved: 1000
Ursolic acid properties retrieved: 1000
Apigenin properties retrieved: 1000
Piperine properties retrieved: 1000
Piperlongumine properties retrieved: 947
Baicalein properties retrieved: 1000
Shikonin properties retrieved: 757
Magnolol properties retrieved: 1000
Honokiol properties retrieved: 1000
Wogonin properties retrieved: 1000
Harmine properties retrieved: 1000
Artemisinin properties retrieved: 662
Thymoquinone properties retrieved: 207
Andrographolide properties retrieved: 597
Diosgenin properties retrieved: 805
Capsaicin properties retrieved: 1000
Eugenol properties ret

## 9. Chemical structure QC — same core filters as the documented 24-parent ZIP workflow

In [9]:
def qc_one_record(rec):
    smi = rec.get("IsomericSMILES") or rec.get("CanonicalSMILES")
    mol = mol_from_smiles(smi)
    if mol is None:
        return False, "invalid_or_missing_smiles", None
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    if "C" not in atoms:
        return False, "no_carbon", mol
    if any(a in METAL_ELEMENTS for a in atoms):
        return False, "metal_containing", mol
    if any(a not in PERMITTED_ELEMENTS for a in atoms):
        return False, "unsupported_element", mol
    if len(Chem.GetMolFrags(mol)) != 1:
        return False, "multi_fragment", mol
    if any(a.GetIsotope() != 0 for a in mol.GetAtoms()):
        return False, "isotopically_labelled", mol
    return True, "PASS", mol

qc_summary = []
qc_records_by_parent = {}
qc_record_audit_rows = []

for row in PANEL:
    parent_cid = int(row["pubchem_cid"])
    records = candidate_records[parent_cid]
    accepted, rejected = [], []
    seen = set()
    missing_cid_count = 0
    duplicate_cid_count = 0

    for rec in records:
        cid = rec.get("CID")
        if cid is None:
            missing_cid_count += 1
            qc_record_audit_rows.append({
                "Parent_Ligand": row["name"], "CID": None,
                "QC_Status": "NOT_EVALUATED", "QC_Reason": "MISSING_CID"
            })
            continue
        cid = int(cid)
        if cid in seen:
            duplicate_cid_count += 1
            qc_record_audit_rows.append({
                "Parent_Ligand": row["name"], "CID": cid,
                "QC_Status": "NOT_EVALUATED", "QC_Reason": "DUPLICATE_CID"
            })
            continue
        seen.add(cid)

        passed_qc, reason, mol = qc_one_record(rec)
        item = dict(rec)
        item["Parent_Ligand"] = row["name"]
        item["Parent_CID"] = parent_cid
        item["PubChem_Similarity"] = f">={SIMILARITY_THRESHOLD}% 2D Tanimoto" if cid != parent_cid else "PARENT_RETAINED"
        item["QC_Status"] = "PASS" if passed_qc else "FAIL"
        item["QC_Reason"] = reason

        if passed_qc:
            item["Selected_SMILES"] = Chem.MolToSmiles(
                mol, canonical=True, isomericSmiles=True
            )
            accepted.append(item)
        else:
            rejected.append(item)

        qc_record_audit_rows.append({
            "Parent_Ligand": row["name"], "CID": cid,
            "QC_Status": item["QC_Status"], "QC_Reason": reason
        })

    # The parent must pass the same QC rules as every other record.
    parent_matches = [x for x in accepted if int(x.get("CID", -1)) == parent_cid]
    if not parent_matches:
        parent_record = parent_props.get(parent_cid, {})
        if not parent_record or parent_record.get("lookup_error"):
            raise RuntimeError(
                f"STOP: parent {row['name']} (CID {parent_cid}) has no valid PubChem record."
            )
        parent_pass, parent_reason, parent_mol = qc_one_record(parent_record)
        if not parent_pass:
            raise RuntimeError(
                f"STOP: parent {row['name']} (CID {parent_cid}) failed the same chemical QC "
                f"applied to all candidates: {parent_reason}"
            )
        parent_item = dict(parent_record)
        parent_item.update({
            "CID": parent_cid,
            "Parent_Ligand": row["name"],
            "Parent_CID": parent_cid,
            "PubChem_Similarity": "PARENT_RETAINED",
            "QC_Status": "PASS",
            "QC_Reason": "PASS",
            "Selected_SMILES": Chem.MolToSmiles(
                parent_mol, canonical=True, isomericSmiles=True
            )
        })
        accepted.insert(0, parent_item)

    qc_records_by_parent[parent_cid] = accepted
    pd.DataFrame(accepted).to_csv(
        row["folder"]/"QC_clean"/f"{row['name']}_QC_metadata.csv", index=False
    )
    pd.DataFrame(rejected).to_csv(
        row["folder"]/"raw"/f"{row['name']}_QC_exclusions.csv", index=False
    )

    counts = pd.Series([x["QC_Reason"] for x in rejected]).value_counts().to_dict()
    qc_summary.append({
        "Parent_Ligand": row["name"], "Parent_CID": parent_cid,
        "Raw_property_records": len(records),
        "Unique_CID_records_evaluated": len(seen),
        "QC_Passing": len(accepted), "QC_Failed": len(rejected),
        "Missing_CID_records": missing_cid_count,
        "Duplicate_CID_records": duplicate_cid_count,
        "Invalid_or_missing_SMILES": counts.get("invalid_or_missing_smiles", 0),
        "No_Carbon": counts.get("no_carbon", 0),
        "Metal_Containing": counts.get("metal_containing", 0),
        "Unsupported_Element": counts.get("unsupported_element", 0),
        "Multi_Fragment": counts.get("multi_fragment", 0),
        "Isotope_Labelled": counts.get("isotopically_labelled", 0),
        "Parent_Passed_QC": any(int(x.get("CID", -1)) == parent_cid for x in accepted)
    })

qc_summary_df = pd.DataFrame(qc_summary)
qc_summary_df.to_csv(SUMMARY_ROOT/"Chemical_QC_Summary.csv", index=False)
pd.DataFrame(qc_record_audit_rows).to_csv(
    SUMMARY_ROOT/"Chemical_QC_Record_Audit.csv", index=False
)
display(qc_summary_df)

if not qc_summary_df["Parent_Passed_QC"].all():
    raise RuntimeError("STOP: at least one parent compound did not pass chemical QC.")


[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:57] WARNING: not removing hydrogen atom without neighbors
[08:42:58] WARNING: not r

,Parent_Ligand,Parent_CID,Raw_property_records,Unique_CID_records_evaluated,QC_Passing,QC_Failed,Missing_CID_records,Duplicate_CID_records,Invalid_or_missing_SMILES,No_Carbon,Metal_Containing,Unsupported_Element,Multi_Fragment,Isotope_Labelled,Parent_Passed_QC
0,Resveratrol,445154,1000,1000,717,283,0,0,0,0,29,53,177,24,True
1,Genistin,5281377,1000,1000,994,6,0,0,0,0,0,0,6,0,True
2,Luteolin,5280445,1000,1000,989,11,0,0,0,0,3,0,8,0,True
3,Plumbagin,10205,1000,1000,873,127,0,0,1,0,25,8,87,6,True
4,Withaferin A,265237,970,970,947,23,0,0,0,0,1,15,6,1,True
5,Berberine,2353,1000,1000,430,570,0,0,0,0,1,3,562,4,True
6,Celastrol,122724,454,454,437,17,0,0,0,0,0,0,14,3,True
7,Genistein,5280961,1000,1000,997,3,0,0,0,0,0,0,3,0,True
8,Oleanolic acid,10494,1000,1000,990,10,0,0,0,0,5,0,5,0,True
9,Betulinic acid,64971,1000,1000,975,25,0,0,0,0,3,9,12,1,True


In [17]:
qc_record_audit_rows.append({
    "Parent_Ligand": row["name"],
    "CID": parent_cid,
    "QC_Status": "PASS",
    "QC_Reason": "PASS_PARENT_RETAINED"
})

## 10. Generate 3D structures — ETKDGv3, seed 42, MMFF with UFF fallback

In [15]:
def make_3d_molecule(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None, "INVALID_SMILES", "None"
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = RANDOM_SEED
    status = AllChem.EmbedMolecule(mol, params)
    method = "ETKDGv3"
    if status != 0:
        params.useRandomCoords = True
        params.randomSeed = RANDOM_SEED
        status = AllChem.EmbedMolecule(mol, params)
        method = "ETKDGv3_random_coords_retry"
    if status != 0:
        return None, "EMBED_FAILED", method

    optimization = "NOT_AVAILABLE"
    optimization_status = None
    try:
        if AllChem.MMFFHasAllMoleculeParams(mol):
            optimization_status = AllChem.MMFFOptimizeMolecule(mol, maxIters=1000)
            optimization = "MMFF"
        elif AllChem.UFFHasAllMoleculeParams(mol):
            optimization_status = AllChem.UFFOptimizeMolecule(mol, maxIters=1000)
            optimization = "UFF"
    except Exception as exc:
        return None, f"OPTIMIZATION_ERROR: {exc}", f"{method};{optimization}"

    if optimization_status is None:
        return None, "NO_SUPPORTED_FORCE_FIELD", f"{method};{optimization}"
    if optimization_status != 0:
        return None, f"OPTIMIZATION_NOT_CONVERGED_{optimization}", f"{method};{optimization}"

    conf = mol.GetConformer()
    if conf.GetNumAtoms() != mol.GetNumAtoms():
        return None, "COORDINATE_ATOM_COUNT_MISMATCH", f"{method};{optimization}"
    coords = conf.GetPositions()
    if not (coords.shape == (mol.GetNumAtoms(), 3)) or not __import__("numpy").isfinite(coords).all():
        return None, "INVALID_3D_COORDINATES", f"{method};{optimization}"
    if len(coords) < 2 or float(__import__("numpy").max(__import__("numpy").ptp(coords, axis=0))) <= 1e-6:
        return None, "DEGENERATE_3D_COORDINATES", f"{method};{optimization}"

    return mol, "SUCCESS", f"{method};{optimization}"


In [11]:
# STEP 11 (RESUMABLE): Generate 3D structures and checkpoint each successful SDF record.
# Rerunning this cell skips CIDs already present in their per-parent SDF files.

required_objects = ["PANEL", "qc_records_by_parent", "make_3d_molecule", "SUMMARY_ROOT"]
missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing_objects) +
        ". Run the prerequisite cells in order."
    )

import os, time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from rdkit import Chem

SUMMARY_ROOT = Path(SUMMARY_ROOT)
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = SUMMARY_ROOT / "3D_SDF_Generation_Summary.csv"
FAILURES_CSV = SUMMARY_ROOT / "3D_SDF_Generation_Failures.csv"

# Preserve failures from earlier interrupted runs.
if FAILURES_CSV.exists():
    try:
        sdf_failure_rows = pd.read_csv(FAILURES_CSV).to_dict("records")
    except Exception:
        sdf_failure_rows = []
else:
    sdf_failure_rows = []

def _existing_sdf_inventory(sdf_path, parent_cid):
    """Scan an existing SDF once and return CIDs, counts and parent-presence status."""
    cids, readable, invalid, parent_written = set(), 0, 0, False
    if sdf_path.exists() and sdf_path.stat().st_size:
        for mol in Chem.SDMolSupplier(str(sdf_path), removeHs=False):
            if mol is None:
                invalid += 1
                continue
            readable += 1
            if mol.HasProp("PubChem_CID"):
                try:
                    cid = int(mol.GetProp("PubChem_CID"))
                    cids.add(cid)
                    if cid == parent_cid:
                        parent_written = True
                except (TypeError, ValueError):
                    pass
    return {"cids": cids, "readable": readable,
            "invalid": invalid, "parent_written": parent_written}

def _append_sdf_record(mol, sdf_path):
    """Write one record to a temporary SDF, then append and flush it immediately."""
    tmp = sdf_path.with_name(sdf_path.stem + "._record_tmp.sdf")
    try:
        writer = Chem.SDWriter(str(tmp))
        try:
            writer.write(mol)
        finally:
            writer.close()
        if not tmp.exists() or tmp.stat().st_size == 0:
            raise RuntimeError("RDKit did not produce a temporary SDF record.")
        with open(tmp, "rb") as src:
            data = src.read()
        with open(sdf_path, "ab") as dst:
            if dst.tell():
                dst.write(b"\n")
            dst.write(data)
            dst.flush()
            os.fsync(dst.fileno())
    finally:
        if tmp.exists():
            tmp.unlink()

# Inventory each SDF once. Cached counts avoid repeatedly rereading large files.
progress = {}
for row in PANEL:
    pcid = int(row["pubchem_cid"])
    path = Path(row["folder"]) / "QC_clean" / f"{row['name']}_derivatives_3D_QC_clean.sdf"
    path.parent.mkdir(parents=True, exist_ok=True)
    progress[pcid] = _existing_sdf_inventory(path, pcid)

def _checkpoint_reports():
    rows = []
    for row in PANEL:
        pcid = int(row["pubchem_cid"])
        path = Path(row["folder"]) / "QC_clean" / f"{row['name']}_derivatives_3D_QC_clean.sdf"
        state = progress[pcid]
        failures = [
            f for f in sdf_failure_rows
            if str(f.get("Parent_CID", "")) == str(pcid)
            or str(f.get("Parent_Ligand", "")) == str(row["name"])
        ]
        rows.append({
            "Parent_Ligand": row["name"], "Parent_CID": pcid,
            "QC_Records_Input": len(qc_records_by_parent.get(pcid, [])),
            "3D_Success": state["readable"], "3D_Failed": len(failures),
            "Parent_3D_Written": state["parent_written"],
            "SDF_Path": str(path), "SDF_Exists": path.exists(),
            "SDF_Bytes": path.stat().st_size if path.exists() else 0,
            "Unreadable_SDF_Records": state["invalid"]
        })
    summary = pd.DataFrame(rows)
    failures_df = pd.DataFrame(sdf_failure_rows)
    if failures_df.empty:
        failures_df = pd.DataFrame(columns=[
            "Parent_Ligand", "Parent_CID", "CID", "Failure_Reason", "Method"
        ])
    summary.to_csv(SUMMARY_CSV, index=False)
    failures_df.to_csv(FAILURES_CSV, index=False)
    return summary, failures_df

started = time.time()
total_input = sum(len(qc_records_by_parent.get(int(r["pubchem_cid"]), [])) for r in PANEL)
total_skipped = total_success = total_failed = 0
print(f"Starting 3D generation for {len(PANEL)} parents; {total_input} QC records.")
print("Successful structures are saved to SDF immediately. CSV reports are checkpointed every 10 records.")

sdf_summary_df, sdf_failures_df = _checkpoint_reports()

for parent_index, row in enumerate(PANEL, start=1):
    pcid = int(row["pubchem_cid"])
    records = qc_records_by_parent.get(pcid, [])
    sdf_path = Path(row["folder"]) / "QC_clean" / f"{row['name']}_derivatives_3D_QC_clean.sdf"
    state = progress[pcid]
    pending = [
        rec for rec in records
        if rec.get("CID") is not None and int(rec["CID"]) not in state["cids"]
    ]
    total_skipped += len(records) - len(pending)
    print(
        f"\n[{parent_index}/{len(PANEL)}] {row['name']}: "
        f"{state['readable']} already saved; {len(pending)} remaining."
    )

    for n, rec in enumerate(tqdm(pending, desc=row["name"], unit="molecule"), start=1):
        cid = rec.get("CID")
        smiles = rec.get("Selected_SMILES")
        method = "NOT_STARTED"
        reason = None
        try:
            if cid is None:
                reason = "MISSING_CID"
            elif not smiles:
                reason = "MISSING_SELECTED_SMILES"
            else:
                mol, status, method = make_3d_molecule(smiles)
                if mol is None or status != "SUCCESS":
                    reason = str(status)
                else:
                    mol.SetProp("_Name", f"{row['name']}_CID_{cid}")
                    mol.SetProp("Parent_Ligand", row["name"])
                    mol.SetProp("Parent_CID", str(pcid))
                    mol.SetProp("PubChem_CID", str(int(cid)))
                    mol.SetProp("Selected_SMILES", smiles)
                    mol.SetProp("3D_Method", str(method))
                    _append_sdf_record(mol, sdf_path)
                    state["cids"].add(int(cid))
                    state["readable"] += 1
                    if int(cid) == pcid:
                        state["parent_written"] = True
                    total_success += 1
        except Exception as exc:
            reason = f"EXCEPTION: {type(exc).__name__}: {exc}"

        if reason is not None:
            total_failed += 1
            sdf_failure_rows.append({
                "Parent_Ligand": row["name"], "Parent_CID": pcid,
                "CID": cid, "Failure_Reason": reason, "Method": method
            })

        # SDF checkpoint occurs after every success; reports every 10 attempts.
        if n % 10 == 0 or n == len(pending):
            sdf_summary_df, sdf_failures_df = _checkpoint_reports()

    sdf_summary_df, sdf_failures_df = _checkpoint_reports()
    if not state["parent_written"]:
        print(f"WARNING: parent CID {pcid} is not present in {sdf_path.name}")

elapsed = (time.time() - started) / 60
sdf_summary_df, sdf_failures_df = _checkpoint_reports()
print("\n3D generation pass finished.")
print(f"New successes: {total_success}; new failures: {total_failed}; already saved/skipped: {total_skipped}")
print(f"Elapsed: {elapsed:.1f} minutes")
print(f"Checkpoint summary: {SUMMARY_CSV}")
print(f"Checkpoint failures: {FAILURES_CSV}")
display(sdf_summary_df)

bad_parents = sdf_summary_df[
    (~sdf_summary_df["SDF_Exists"]) |
    (sdf_summary_df["SDF_Bytes"] <= 0) |
    (sdf_summary_df["3D_Success"] <= 0) |
    (~sdf_summary_df["Parent_3D_Written"]) |
    (sdf_summary_df["Unreadable_SDF_Records"] > 0)
]
if len(bad_parents):
    raise RuntimeError(
        "STOP: one or more parent SDFs are missing, empty, unreadable, or lack the parent "
        "conformer. Completed SDF records and checkpoint reports are retained. Fix the "
        "reported cause and rerun this cell; CIDs already present in SDF files are skipped."
    )


Starting 3D generation for 31 parents; 26066 QC records.
Successful structures are saved to SDF immediately. CSV reports are checkpointed every 10 records.

[1/31] Resveratrol: 0 already saved; 717 remaining.


Resveratrol:   0%|          | 0/717 [00:00<?, ?molecule/s]


[2/31] Genistin: 0 already saved; 994 remaining.


Genistin:   0%|          | 0/994 [00:00<?, ?molecule/s]


[3/31] Luteolin: 0 already saved; 989 remaining.


Luteolin:   0%|          | 0/989 [00:00<?, ?molecule/s]

... (56 outputs left out)


[31/31] Myricetin: 0 already saved; 986 remaining.


Myricetin:   0%|          | 0/986 [00:00<?, ?molecule/s]


3D generation pass finished.
New successes: 25896; new failures: 170; already saved/skipped: 0
Elapsed: 304.7 minutes
Checkpoint summary: /content/ZINC_Priority_31_Ligand_Library_FINAL/03_Summary/3D_SDF_Generation_Summary.csv
Checkpoint failures: /content/ZINC_Priority_31_Ligand_Library_FINAL/03_Summary/3D_SDF_Generation_Failures.csv


,Parent_Ligand,Parent_CID,QC_Records_Input,3D_Success,3D_Failed,Parent_3D_Written,SDF_Path,SDF_Exists,SDF_Bytes,Unreadable_SDF_Records
0,Resveratrol,445154,717,717,0,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,2256418,0
1,Genistin,5281377,994,994,0,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,5584757,0
2,Luteolin,5280445,989,989,0,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,3731163,0
3,Plumbagin,10205,873,871,2,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,2439873,0
4,Withaferin A,265237,947,946,1,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,6269185,0
5,Berberine,2353,430,430,0,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,1953460,0
6,Celastrol,122724,437,437,0,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,2917940,0
7,Genistein,5280961,997,997,0,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,3796540,0
8,Oleanolic acid,10494,990,988,2,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,7448934,0
9,Betulinic acid,64971,975,973,2,True,/content/ZINC_Priority_31_Ligand_Library_FINAL...,True,7753019,0


In [18]:
# STEP 11B: Repair extra blank lines between SDF records.
# Original SDF files are backed up before modification.

from pathlib import Path
import shutil

repair_report = []

for row in PANEL:
    sdf_path = (
        Path(row["folder"]) / "QC_clean" /
        f"{row['name']}_derivatives_3D_QC_clean.sdf"
    )

    if not sdf_path.exists() or sdf_path.stat().st_size == 0:
        repair_report.append({
            "Parent_Ligand": row["name"],
            "Status": "MISSING_OR_EMPTY"
        })
        continue

    backup_path = sdf_path.with_suffix(".original_backup.sdf")
    if not backup_path.exists():
        shutil.copy2(sdf_path, backup_path)

    with open(sdf_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    repaired_lines = []
    skip_blank_after_delimiter = False
    removed_blank_lines = 0

    for line in lines:
        if skip_blank_after_delimiter and not line.strip():
            removed_blank_lines += 1
            continue

        repaired_lines.append(line)
        skip_blank_after_delimiter = line.strip() == "$$$$"

    temp_path = sdf_path.with_suffix(".repaired_tmp.sdf")

    with open(temp_path, "w", encoding="utf-8", newline="") as f:
        f.writelines(repaired_lines)

    temp_path.replace(sdf_path)

    repair_report.append({
        "Parent_Ligand": row["name"],
        "Removed_Blank_Separators": removed_blank_lines,
        "Original_Backup": str(backup_path),
        "Repaired_SDF_Bytes": sdf_path.stat().st_size,
        "Status": "REPAIRED"
    })

repair_report_df = pd.DataFrame(repair_report)
display(repair_report_df)

repair_report_df.to_csv(
    SUMMARY_ROOT / "SDF_Separator_Repair_Report.csv",
    index=False
)

,Parent_Ligand,Removed_Blank_Separators,Original_Backup,Repaired_SDF_Bytes,Status
0,Resveratrol,716,/content/ZINC_Priority_31_Ligand_Library_FINAL...,2255702,REPAIRED
1,Genistin,993,/content/ZINC_Priority_31_Ligand_Library_FINAL...,5583764,REPAIRED
2,Luteolin,988,/content/ZINC_Priority_31_Ligand_Library_FINAL...,3730175,REPAIRED
3,Plumbagin,870,/content/ZINC_Priority_31_Ligand_Library_FINAL...,2439003,REPAIRED
4,Withaferin A,945,/content/ZINC_Priority_31_Ligand_Library_FINAL...,6268240,REPAIRED
5,Berberine,429,/content/ZINC_Priority_31_Ligand_Library_FINAL...,1953031,REPAIRED
6,Celastrol,436,/content/ZINC_Priority_31_Ligand_Library_FINAL...,2917504,REPAIRED
7,Genistein,996,/content/ZINC_Priority_31_Ligand_Library_FINAL...,3795544,REPAIRED
8,Oleanolic acid,987,/content/ZINC_Priority_31_Ligand_Library_FINAL...,7447947,REPAIRED
9,Betulinic acid,972,/content/ZINC_Priority_31_Ligand_Library_FINAL...,7752047,REPAIRED


## 12. Read-back verification and 3D conformer audit


In [19]:
# STEP 12: Read each SDF back and verify readable records and valid 3D coordinates.
if "sdf_summary_df" not in globals():
    raise RuntimeError(
        "SDF generation has not completed in this runtime. Run the preceding "
        "'Generate 3D structures and write the per-parent SDF files' cell first. "
        "If the runtime was reset, run the notebook from the beginning."
    )

verification_rows = []
import numpy as np
from rdkit import Chem

for row in PANEL:
    sdf_path = row["folder"] / "QC_clean" / f"{row['name']}_derivatives_3D_QC_clean.sdf"
    generated_match = sdf_summary_df.loc[
        sdf_summary_df["Parent_Ligand"] == row["name"], "3D_Success"
    ]
    generated = int(generated_match.iloc[0]) if not generated_match.empty else 0

    readable, all_3d, unreadable = 0, True, 0
    if not sdf_path.exists() or sdf_path.stat().st_size == 0:
        all_3d = False
        unreadable = generated if generated > 0 else 1
    else:
        supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
        for mol in supplier:
            if mol is None:
                unreadable += 1
                all_3d = False
                continue
            readable += 1
            if mol.GetNumConformers() < 1:
                all_3d = False
                continue
            coords = np.asarray(mol.GetConformer().GetPositions(), dtype=float)
            if coords.shape != (mol.GetNumAtoms(), 3) or not np.isfinite(coords).all():
                all_3d = False
            elif len(coords) < 2 or float(np.max(np.ptp(coords, axis=0))) <= 1e-6:
                all_3d = False

    verification_rows.append({
        "Parent_Ligand": row["name"], "SDF_Exists": sdf_path.exists(),
        "SDF_Generated_Records": generated, "SDF_Readable_Records": readable,
        "Unreadable_Records": unreadable,
        "All_Readable_Records_Have_Valid_3D_Coordinates": all_3d,
        "Readback_Match": generated == readable and unreadable == 0 and generated > 0
    })

verification_df = pd.DataFrame(verification_rows)
verification_df.to_csv(SUMMARY_ROOT/"SDF_Readback_Verification.csv", index=False)
display(verification_df)

if (
    not verification_df["SDF_Exists"].all()
    or not verification_df["All_Readable_Records_Have_Valid_3D_Coordinates"].all()
    or not verification_df["Readback_Match"].all()
):
    raise RuntimeError(
        "STOP: at least one SDF is missing, empty, unreadable, or failed 3D-coordinate "
        "verification. Inspect the generation and read-back CSV reports."
    )

,Parent_Ligand,SDF_Exists,SDF_Generated_Records,SDF_Readable_Records,Unreadable_Records,All_Readable_Records_Have_Valid_3D_Coordinates,Readback_Match
0,Resveratrol,True,717,717,0,True,True
1,Genistin,True,994,994,0,True,True
2,Luteolin,True,989,989,0,True,True
3,Plumbagin,True,871,871,0,True,True
4,Withaferin A,True,946,946,0,True,True
5,Berberine,True,430,430,0,True,True
6,Celastrol,True,437,437,0,True,True
7,Genistein,True,997,997,0,True,True
8,Oleanolic acid,True,988,988,0,True,True
9,Betulinic acid,True,973,973,0,True,True


## 13. PubMed literature retrieval (automated discovery, not a systematic review)

In [20]:
# ============================================================
# STEP 13: PUBMED CANCER LITERATURE RETRIEVAL
# Automated literature discovery; NOT a systematic review.
# ============================================================

import time
import pandas as pd
from pathlib import Path

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

pubmed_rows = []
pubmed_search_audit = []

for row in PANEL:

    compound = row["name"]

    term = (
        f'"{compound}"[Title/Abstract] AND '
        f'(cancer[Title/Abstract] OR '
        f'tumour[Title/Abstract] OR '
        f'tumor[Title/Abstract] OR '
        f'carcinoma[Title/Abstract] OR '
        f'neoplasm[Title/Abstract])'
    )

    print(f"\nSearching PubMed: {compound}")

    try:
        # ----------------------------------------------------
        # 1. PubMed search
        # ----------------------------------------------------

        search_response = request_with_retries(
            "GET",
            f"{EUTILS}/esearch.fcgi",
            params={
                "db": "pubmed",
                "term": term,
                "retmax": 10,
                "retstart": 0,
                "sort": "relevance",
                "retmode": "json"
            }
        )

        search_data = search_response.json().get(
            "esearchresult", {}
        )

        total_count = int(search_data.get("count", 0))
        pmids = search_data.get("idlist", [])
        query_translation = search_data.get(
            "querytranslation", ""
        )

        time.sleep(0.4)

        # ----------------------------------------------------
        # 2. Record search audit
        # ----------------------------------------------------

        pubmed_search_audit.append({
            "Compound": compound,
            "Search_Term": term,
            "Total_PubMed_Results": total_count,
            "Retrieved_Records": len(pmids),
            "Query_Translation": query_translation,
            "Search_Status": "SUCCESS"
        })

        # ----------------------------------------------------
        # 3. No-result handling
        # ----------------------------------------------------

        if not pmids:

            pubmed_rows.append({
                "Compound": compound,
                "Search_Term": term,
                "Total_PubMed_Results": total_count,
                "PMID": "",
                "Title": "",
                "Journal": "",
                "PubDate": "",
                "DOI": "",
                "PubMed_URL": "",
                "Retrieval_Status": "NO_RESULTS"
            })

            continue

        # ----------------------------------------------------
        # 4. Retrieve article metadata
        # ----------------------------------------------------

        summary_response = request_with_retries(
            "GET",
            f"{EUTILS}/esummary.fcgi",
            params={
                "db": "pubmed",
                "id": ",".join(pmids),
                "retmode": "json"
            }
        )

        summary_data = summary_response.json().get(
            "result", {}
        )

        time.sleep(0.4)

        # ----------------------------------------------------
        # 5. Extract article information
        # ----------------------------------------------------

        for pmid in pmids:

            item = summary_data.get(pmid, {})

            if not item:
                pubmed_rows.append({
                    "Compound": compound,
                    "Search_Term": term,
                    "Total_PubMed_Results": total_count,
                    "PMID": pmid,
                    "Title": "",
                    "Journal": "",
                    "PubDate": "",
                    "DOI": "",
                    "PubMed_URL":
                        f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                    "Retrieval_Status": "METADATA_UNAVAILABLE"
                })
                continue

            doi = ""

            for aid in item.get("articleids", []):

                if aid.get("idtype") == "doi":
                    doi = aid.get("value", "")
                    break

            pubmed_rows.append({
                "Compound": compound,
                "Search_Term": term,
                "Total_PubMed_Results": total_count,
                "PMID": pmid,
                "Title": item.get("title", ""),
                "Journal": item.get("fulljournalname", ""),
                "PubDate": item.get("pubdate", ""),
                "DOI": doi,
                "PubMed_URL":
                    f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                "Retrieval_Status": "SUCCESS"
            })

        print(
            f"Results: {total_count} | "
            f"Retrieved: {len(pmids)}"
        )

    except Exception as exc:

        pubmed_search_audit.append({
            "Compound": compound,
            "Search_Term": term,
            "Total_PubMed_Results": None,
            "Retrieved_Records": 0,
            "Query_Translation": "",
            "Search_Status": f"ERROR: {type(exc).__name__}: {exc}"
        })

        pubmed_rows.append({
            "Compound": compound,
            "Search_Term": term,
            "Total_PubMed_Results": None,
            "PMID": "",
            "Title": str(exc),
            "Journal": "",
            "PubDate": "",
            "DOI": "",
            "PubMed_URL": "",
            "Retrieval_Status": "SEARCH_ERROR"
        })

    time.sleep(0.4)


# ============================================================
# 6. SAVE OUTPUTS
# ============================================================

pubmed_df = pd.DataFrame(pubmed_rows)

pubmed_audit_df = pd.DataFrame(
    pubmed_search_audit
)

pubmed_df.to_csv(
    SUMMARY_ROOT / "PubMed_Cancer_Literature_Search.csv",
    index=False
)

pubmed_audit_df.to_csv(
    SUMMARY_ROOT / "PubMed_Search_Audit.csv",
    index=False
)

print("\n========== PUBMED SEARCH SUMMARY ==========")

print("Compounds searched:", len(PANEL))
print("Article records retrieved:", len(pubmed_df))
print(
    "Total matching results across compound searches:",
    pubmed_audit_df["Total_PubMed_Results"].sum()
)

print(
    "Successful searches:",
    (pubmed_audit_df["Search_Status"] == "SUCCESS").sum()
)

print(
    "Failed searches:",
    (pubmed_audit_df["Search_Status"] != "SUCCESS").sum()
)

print("\nSaved files:")
print(SUMMARY_ROOT / "PubMed_Cancer_Literature_Search.csv")
print(SUMMARY_ROOT / "PubMed_Search_Audit.csv")

display(pubmed_audit_df)
display(pubmed_df.head(20))


Searching PubMed: Resveratrol
Results: 5666 | Retrieved: 10

Searching PubMed: Genistin
Results: 90 | Retrieved: 10

Searching PubMed: Luteolin
Results: 1792 | Retrieved: 10

Searching PubMed: Plumbagin
Results: 388 | Retrieved: 10

Searching PubMed: Withaferin A
Results: 481 | Retrieved: 10

Searching PubMed: Berberine
Results: 1749 | Retrieved: 10

Searching PubMed: Celastrol
Results: 687 | Retrieved: 10

Searching PubMed: Genistein
Results: 3264 | Retrieved: 10

Searching PubMed: Oleanolic acid
Results: 838 | Retrieved: 10

Searching PubMed: Betulinic acid
Results: 778 | Retrieved: 10

Searching PubMed: Kaempferol
Results: 1799 | Retrieved: 10

Searching PubMed: Ursolic acid
Results: 1048 | Retrieved: 10

Searching PubMed: Apigenin
Results: 1670 | Retrieved: 10

Searching PubMed: Piperine
Results: 449 | Retrieved: 10

Searching PubMed: Piperlongumine
Results: 322 | Retrieved: 10

Searching PubMed: Baicalein
Results: 877 | Retrieved: 10

Searching PubMed: Shikonin
Results: 645 | Ret

,Compound,Search_Term,Total_PubMed_Results,Retrieved_Records,Query_Translation,Search_Status
0,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,10,"""Resveratrol""[Title/Abstract] AND (""cancer""[Ti...",SUCCESS
1,Genistin,"""Genistin""[Title/Abstract] AND (cancer[Title/A...",90,10,"""Genistin""[Title/Abstract] AND (""cancer""[Title...",SUCCESS
2,Luteolin,"""Luteolin""[Title/Abstract] AND (cancer[Title/A...",1792,10,"""Luteolin""[Title/Abstract] AND (""cancer""[Title...",SUCCESS
3,Plumbagin,"""Plumbagin""[Title/Abstract] AND (cancer[Title/...",388,10,"""Plumbagin""[Title/Abstract] AND (""cancer""[Titl...",SUCCESS
4,Withaferin A,"""Withaferin A""[Title/Abstract] AND (cancer[Tit...",481,10,"""Withaferin A""[Title/Abstract] AND (""cancer""[T...",SUCCESS
5,Berberine,"""Berberine""[Title/Abstract] AND (cancer[Title/...",1749,10,"""Berberine""[Title/Abstract] AND (""cancer""[Titl...",SUCCESS
6,Celastrol,"""Celastrol""[Title/Abstract] AND (cancer[Title/...",687,10,"""Celastrol""[Title/Abstract] AND (""cancer""[Titl...",SUCCESS
7,Genistein,"""Genistein""[Title/Abstract] AND (cancer[Title/...",3264,10,"""Genistein""[Title/Abstract] AND (""cancer""[Titl...",SUCCESS
8,Oleanolic acid,"""Oleanolic acid""[Title/Abstract] AND (cancer[T...",838,10,"""Oleanolic acid""[Title/Abstract] AND (""cancer""...",SUCCESS
9,Betulinic acid,"""Betulinic acid""[Title/Abstract] AND (cancer[T...",778,10,"""Betulinic acid""[Title/Abstract] AND (""cancer""...",SUCCESS


,Compound,Search_Term,Total_PubMed_Results,PMID,Title,Journal,PubDate,DOI,PubMed_URL,Retrieval_Status
0,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,20641411,"5-[(E)-2-(4-[(18)F]Fluorophenyl)ethenyl]-1,3-b...",,2004,,https://pubmed.ncbi.nlm.nih.gov/20641411/,SUCCESS
1,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,32961987,Resveratrol and Tumor Microenvironment: Mechan...,"Molecules (Basel, Switzerland)",2020 Sep 18,10.3390/molecules25184282,https://pubmed.ncbi.nlm.nih.gov/32961987/,SUCCESS
2,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,35858935,Curcumin- and resveratrol-co-loaded nanopartic...,Journal of nanobiotechnology,2022 Jul 20,10.1186/s12951-022-01554-y,https://pubmed.ncbi.nlm.nih.gov/35858935/,SUCCESS
3,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,34171623,Resveratrol for targeting the tumor microenvir...,International immunopharmacology,2021 Sep,10.1016/j.intimp.2021.107895,https://pubmed.ncbi.nlm.nih.gov/34171623/,SUCCESS
4,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,37862428,"Resveratrol: biology, metabolism, and detrimen...",Nutrition reviews,2024 Oct 1,10.1093/nutrit/nuad133,https://pubmed.ncbi.nlm.nih.gov/37862428/,SUCCESS
5,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,38190940,Resveratrol and p53: How are they involved in ...,Journal of advanced research,2024 Dec,10.1016/j.jare.2024.01.005,https://pubmed.ncbi.nlm.nih.gov/38190940/,SUCCESS
6,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,38026933,Molecular mechanisms of resveratrol as chemo a...,Frontiers in pharmacology,2023,10.3389/fphar.2023.1287505,https://pubmed.ncbi.nlm.nih.gov/38026933/,SUCCESS
7,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,39761424,Resveratrol amplifies the anti-tumor effect of...,Gut microbes,2025 Dec,10.1080/19490976.2024.2447821,https://pubmed.ncbi.nlm.nih.gov/39761424/,SUCCESS
8,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,37592772,Resveratrol: Targeting Cancer Stem Cells and n...,Current molecular medicine,2024,10.2174/1566524023666230817102114,https://pubmed.ncbi.nlm.nih.gov/37592772/,SUCCESS
9,Resveratrol,"""Resveratrol""[Title/Abstract] AND (cancer[Titl...",5666,32233996,Focus on the Use of Resveratrol as an Adjuvant...,Current pharmaceutical design,2020,10.2174/1381612826666200401085634,https://pubmed.ncbi.nlm.nih.gov/32233996/,SUCCESS


## 14. ChEMBL cross-reference checks and BindingDB lookup status

In [21]:
import re
import json
import time
import pandas as pd
from pathlib import Path

# ============================================================
# PART A: ChEMBL ID CROSS-CHECK
# ============================================================

chembl_rows = []

for row in PANEL:
    url = (
        f"https://www.ebi.ac.uk/chembl/api/data/"
        f"molecule/{row['chembl_id']}.json"
    )

    try:
        resp = request_with_retries("GET", url)
        resp.raise_for_status()
        js = resp.json()

        resolved_id = js.get("molecule_chembl_id")
        pref_name = js.get("pref_name")

        chembl_rows.append({
            "Compound": row["name"],
            "ChEMBL_ID_Provided": row["chembl_id"],
            "ChEMBL_Resolved_ID": resolved_id,
            "Pref_Name": pref_name,
            "Max_Phase": js.get("max_phase"),
            "ID_Match": resolved_id == row["chembl_id"],
            "ChEMBL_Status": (
                "RESOLVED"
                if resolved_id == row["chembl_id"]
                else "ID_MISMATCH"
            )
        })

    except Exception as exc:
        chembl_rows.append({
            "Compound": row["name"],
            "ChEMBL_ID_Provided": row["chembl_id"],
            "ChEMBL_Resolved_ID": "",
            "Pref_Name": "",
            "Max_Phase": "",
            "ID_Match": False,
            "ChEMBL_Status": (
                f"LOOKUP_FAILED: {type(exc).__name__}: {exc}"
            )
        })

    time.sleep(0.25)


chembl_df = pd.DataFrame(chembl_rows)

chembl_df.to_csv(
    SUMMARY_ROOT / "ChEMBL_ID_Crosscheck.csv",
    index=False
)


# ============================================================
# PART B: BindingDB STRUCTURE-SIMILARITY LOOKUP
# ============================================================

def walk_dicts(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk_dicts(value)

    elif isinstance(obj, list):
        for value in obj:
            yield from walk_dicts(value)


def get_bindingdb_value(record, *keys):
    low = {str(k).lower(): v for k, v in record.items()}

    for key in keys:
        value = low.get(key.lower())

        if value is not None and value != "":
            return value

    return ""


bindingdb_rows = []
bindingdb_audit_rows = []

for row in PANEL:

    compound = row["name"]
    cid = int(row["pubchem_cid"])

    parent = parent_props.get(cid, {})

    parent_smi = (
        parent.get("IsomericSMILES")
        or parent.get("CanonicalSMILES")
    )

    safe_name = re.sub(
        r"[^A-Za-z0-9]+", "_", compound
    )

    raw_path = SUMMARY_ROOT / f"BindingDB_raw_{safe_name}.txt"

    if not parent_smi:
        bindingdb_audit_rows.append({
            "Compound": compound,
            "PubChem_CID": cid,
            "Lookup_Status": "MISSING_PARENT_SMILES",
            "Returned_Records": 0
        })
        continue

    parent_canonical = canonical_isomeric(parent_smi)

    try:

        url = "https://bindingdb.org/rest/getTargetByCompound"

        resp = request_with_retries(
            "GET",
            url,
            params={
                "smiles": parent_smi,
                "cutoff": 0.85,
                "response": "application/json"
            }
        )

        resp.raise_for_status()

        raw_path.write_text(
            resp.text,
            encoding="utf-8"
        )

        try:
            payload = resp.json()
        except Exception:
            payload = None

        records = []

        if payload is not None:
            for obj in walk_dicts(payload):

                smiles = get_bindingdb_value(
                    obj,
                    "smiles",
                    "smile",
                    "bdb.smiles"
                )

                if smiles:
                    records.append(obj)

        # Remove duplicate JSON records.
        unique_records = []
        seen_records = set()

        for obj in records:
            signature = json.dumps(
                obj,
                sort_keys=True,
                default=str
            )

            if signature not in seen_records:
                seen_records.add(signature)
                unique_records.append(obj)

        records = unique_records

        exact_count = 0
        similar_count = 0

        for obj in records:

            returned_smiles = get_bindingdb_value(
                obj,
                "smiles",
                "smile",
                "bdb.smiles"
            )

            returned_canonical = ""

            if returned_smiles:
                try:
                    returned_canonical = canonical_isomeric(
                        str(returned_smiles)
                    )
                except Exception:
                    returned_canonical = ""

            exact_match = (
                bool(returned_canonical)
                and returned_canonical == parent_canonical
            )

            if exact_match:
                exact_count += 1
                record_class = "EXACT_PARENT_STRUCTURE"
            else:
                similar_count += 1
                record_class = "SIMILAR_STRUCTURE_ONLY"

            bindingdb_rows.append({
                "Compound": compound,
                "PubChem_CID": cid,
                "Returned_SMILES": returned_smiles,
                "Exact_parent_structure_match": exact_match,
                "BindingDB_Monomer_ID": get_bindingdb_value(
                    obj, "monomerid", "bdb.monomerid"
                ),
                "Target": get_bindingdb_value(
                    obj, "target", "query", "bdb.target"
                ),
                "Species": get_bindingdb_value(
                    obj, "species", "bdb.species"
                ),
                "Affinity_Type": get_bindingdb_value(
                    obj, "affinity_type", "bdb.affinity_type"
                ),
                "Affinity": get_bindingdb_value(
                    obj, "affinity", "bdb.affinity"
                ),
                "PMID": get_bindingdb_value(
                    obj, "pmid", "bdb.pmid"
                ),
                "DOI": get_bindingdb_value(
                    obj, "doi", "bdb.doi"
                ),
                "Record_Class": record_class
            })

        if not records:
            bindingdb_rows.append({
                "Compound": compound,
                "PubChem_CID": cid,
                "Returned_SMILES": "",
                "Exact_parent_structure_match": False,
                "BindingDB_Monomer_ID": "",
                "Target": "",
                "Species": "",
                "Affinity_Type": "",
                "Affinity": "",
                "PMID": "",
                "DOI": "",
                "Record_Class": "NO_PARSEABLE_MATCH_OR_EMPTY_RESPONSE"
            })

        bindingdb_audit_rows.append({
            "Compound": compound,
            "PubChem_CID": cid,
            "HTTP_Status": resp.status_code,
            "Returned_Records": len(records),
            "Exact_Structure_Matches": exact_count,
            "Similar_Structure_Records": similar_count,
            "Lookup_Status": (
                "RECORDS_FOUND" if records
                else "NO_PARSEABLE_MATCH_OR_EMPTY_RESPONSE"
            ),
            "Raw_Response_File": str(raw_path)
        })

    except Exception as exc:

        raw_path.write_text(
            str(exc),
            encoding="utf-8"
        )

        bindingdb_rows.append({
            "Compound": compound,
            "PubChem_CID": cid,
            "Returned_SMILES": "",
            "Exact_parent_structure_match": False,
            "BindingDB_Monomer_ID": "",
            "Target": "",
            "Species": "",
            "Affinity_Type": "",
            "Affinity": "",
            "PMID": "",
            "DOI": "",
            "Record_Class": f"LOOKUP_FAILED: {exc}"
        })

        bindingdb_audit_rows.append({
            "Compound": compound,
            "PubChem_CID": cid,
            "HTTP_Status": "",
            "Returned_Records": 0,
            "Exact_Structure_Matches": 0,
            "Similar_Structure_Records": 0,
            "Lookup_Status": f"LOOKUP_FAILED: {exc}",
            "Raw_Response_File": str(raw_path)
        })

    time.sleep(1.0)


# ============================================================
# PART C: SAVE REPORTS
# ============================================================

bindingdb_df = pd.DataFrame(bindingdb_rows)

bindingdb_audit_df = pd.DataFrame(
    bindingdb_audit_rows
)

chembl_df.to_csv(
    SUMMARY_ROOT / "ChEMBL_ID_Crosscheck.csv",
    index=False
)

bindingdb_df.to_csv(
    SUMMARY_ROOT / "BindingDB_Structure_Match_and_Assay_Audit.csv",
    index=False
)

bindingdb_audit_df.to_csv(
    SUMMARY_ROOT / "BindingDB_Lookup_Audit.csv",
    index=False
)

print("ChEMBL resolved:",
      (chembl_df["ChEMBL_Status"] == "RESOLVED").sum(), "/ 31")

print("BindingDB searches:",
      len(bindingdb_audit_df), "/ 31")

display(chembl_df)
display(bindingdb_audit_df)
display(bindingdb_df.head(30))

ChEMBL resolved: 31 / 31
BindingDB searches: 31 / 31


,Compound,ChEMBL_ID_Provided,ChEMBL_Resolved_ID,Pref_Name,Max_Phase,ID_Match,ChEMBL_Status
0,Resveratrol,CHEMBL165,CHEMBL165,RESVERATROL,3.0,True,RESOLVED
1,Genistin,CHEMBL486625,CHEMBL486625,GENISTIN,None,True,RESOLVED
2,Luteolin,CHEMBL151,CHEMBL151,LUTEOLIN,2.0,True,RESOLVED
3,Plumbagin,CHEMBL295316,CHEMBL295316,PLUMBAGIN,1.0,True,RESOLVED
4,Withaferin A,CHEMBL517080,CHEMBL517080,WITHAFERIN A,1.0,True,RESOLVED
5,Berberine,CHEMBL295124,CHEMBL295124,BERBERINE,4.0,True,RESOLVED
6,Celastrol,CHEMBL301982,CHEMBL301982,CELASTROL,None,True,RESOLVED
7,Genistein,CHEMBL44,CHEMBL44,GENISTEIN,2.0,True,RESOLVED
8,Oleanolic acid,CHEMBL168,CHEMBL168,OLEANOLIC_ACID,None,True,RESOLVED
9,Betulinic acid,CHEMBL269277,CHEMBL269277,BETULINIC ACID,1.0,True,RESOLVED


,Compound,PubChem_CID,HTTP_Status,Returned_Records,Exact_Structure_Matches,Similar_Structure_Records,Lookup_Status,Raw_Response_File
0,Resveratrol,445154,200,20,20,0,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...
1,Genistin,5281377,200,0,0,0,NO_PARSEABLE_MATCH_OR_EMPTY_RESPONSE,/content/ZINC_Priority_31_Ligand_Library_FINAL...
2,Luteolin,5280445,200,29,29,0,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...
3,Plumbagin,10205,200,3,3,0,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...
4,Withaferin A,265237,200,2,1,1,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...
5,Berberine,2353,200,8,7,1,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...
6,Celastrol,122724,200,3,3,0,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...
7,Genistein,5280961,200,23,23,0,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...
8,Oleanolic acid,10494,200,14,7,7,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...
9,Betulinic acid,64971,200,14,11,3,RECORDS_FOUND,/content/ZINC_Priority_31_Ligand_Library_FINAL...


,Compound,PubChem_CID,Returned_SMILES,Exact_parent_structure_match,BindingDB_Monomer_ID,Target,Species,Affinity_Type,Affinity,PMID,DOI,Record_Class
0,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Alpha-1D adrenergic receptor,Rat,IC50,19950,,,EXACT_PARENT_STRUCTURE
1,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Amine oxidase [flavin-containing] B,Human,IC50,5010,,,EXACT_PARENT_STRUCTURE
2,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Amyloid-beta precursor protein,Human,Kd,38500,,,EXACT_PARENT_STRUCTURE
3,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Carbonic anhydrase 12,Human,Ki,950,,,EXACT_PARENT_STRUCTURE
4,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Carbonic anhydrase 7,Human,Ki,4350,,,EXACT_PARENT_STRUCTURE
5,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Cathepsin D,Human,IC50,138000,,,EXACT_PARENT_STRUCTURE
6,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Cytochrome P450 1B1,Human,Ki,177,,,EXACT_PARENT_STRUCTURE
7,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Dipeptidyl peptidase 4,Human,IC50,0.600000,,,EXACT_PARENT_STRUCTURE
8,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Estrogen receptor,Human,Ki,785,,,EXACT_PARENT_STRUCTURE
9,Resveratrol,445154,c1cc(ccc1\C=C\c2cc(cc(c2)O)O)O,True,23926,Lysosomal alpha-glucosidase,Rat,IC50,>400000,,,EXACT_PARENT_STRUCTURE


## 15. Consolidated 31-parent summary

In [22]:
summary_df = panel_df.copy()

summary_df = summary_df.merge(
    parent_audit_df,
    left_on="name",
    right_on="Compound",
    how="left",
    validate="one_to_one"
)

summary_df = summary_df.merge(
    retrieval_df,
    left_on="name",
    right_on="Parent_Ligand",
    how="left",
    suffixes=("", "_retrieval"),
    validate="one_to_one"
)

summary_df = summary_df.merge(
    qc_summary_df,
    left_on="name",
    right_on="Parent_Ligand",
    how="left",
    suffixes=("", "_qc"),
    validate="one_to_one"
)

summary_df = summary_df.merge(
    sdf_summary_df,
    left_on="name",
    right_on="Parent_Ligand",
    how="left",
    suffixes=("", "_3d"),
    validate="one_to_one"
)

summary_df = summary_df.merge(
    verification_df,
    left_on="name",
    right_on="Parent_Ligand",
    how="left",
    suffixes=("", "_verify"),
    validate="one_to_one"
)

# Save final summary
output_file = SUMMARY_ROOT / "ALL_31_ZINC_PRIORITY_PARENTS_FINAL_SUMMARY.csv"
summary_df.to_csv(output_file, index=False)

print("Final summary shape:", summary_df.shape)
print("Unique parent compounds:", summary_df["name"].nunique())
print("Missing values per column:")
display(summary_df.isna().sum().to_frame("Missing"))

display(summary_df)

Final summary shape: (31, 56)
Unique parent compounds: 31
Missing values per column:


,Missing
name,0
zinc_id,0
pubchem_cid,0
chembl_id,0
origin,0
pubmed_start,0
Compound,0
ZINC_ID_Cross_Reference,0
ZINC22_Live_Status,0
PubChem_CID,0


,name,zinc_id,pubchem_cid,chembl_id,origin,pubmed_start,Compound,ZINC_ID_Cross_Reference,ZINC22_Live_Status,PubChem_CID,...,SDF_Exists,SDF_Bytes,Unreadable_SDF_Records,Parent_Ligand_verify,SDF_Exists_verify,SDF_Generated_Records,SDF_Readable_Records,Unreadable_Records,All_Readable_Records_Have_Valid_3D_Coordinates,Readback_Match
0,Resveratrol,ZINC000000006787,445154,CHEMBL165,Vitis vinifera; Reynoutria japonica (syn. Poly...,32449796;36499286,Resveratrol,ZINC000000006787,NOT_QUERIED_SKIPPED_AS_REQUESTED,445154,...,True,2256418,0,Resveratrol,True,717,717,0,True,True
1,Genistin,ZINC000004097913,5281377,CHEMBL486625,Glycine max (soybean) and other legumes,31192770,Genistin,ZINC000004097913,NOT_QUERIED_SKIPPED_AS_REQUESTED,5281377,...,True,5584757,0,Genistin,True,994,994,0,True,True
2,Luteolin,ZINC000018185774,5280445,CHEMBL151,"Parsley, celery, chamomile and numerous plants",36499286;31192770,Luteolin,ZINC000018185774,NOT_QUERIED_SKIPPED_AS_REQUESTED,5280445,...,True,3731163,0,Luteolin,True,989,989,0,True,True
3,Plumbagin,ZINC000000058187,10205,CHEMBL295316,"Plumbago zeylanica and other Plumbaginaceae, E...",28359198;33344645,Plumbagin,ZINC000000058187,NOT_QUERIED_SKIPPED_AS_REQUESTED,10205,...,True,2439873,0,Plumbagin,True,871,871,0,True,True
4,Withaferin A,ZINC000008234189,265237,CHEMBL517080,"Withania somnifera (ashwagandha), particularly...",32727824;37259311;33434782,Withaferin A,ZINC000008234189,NOT_QUERIED_SKIPPED_AS_REQUESTED,265237,...,True,6269185,0,Withaferin A,True,946,946,0,True,True
5,Berberine,ZINC000003779067,2353,CHEMBL295124,"Berberis spp., Coptis chinensis and other Berb...",32449796;21777476,Berberine,ZINC000003779067,NOT_QUERIED_SKIPPED_AS_REQUESTED,2353,...,True,1953460,0,Berberine,True,430,430,0,True,True
6,Celastrol,ZINC000019795938,122724,CHEMBL301982,Tripterygium wilfordii roots and other Celastr...,33364939;37196541,Celastrol,ZINC000019795938,NOT_QUERIED_SKIPPED_AS_REQUESTED,122724,...,True,2917940,0,Celastrol,True,437,437,0,True,True
7,Genistein,ZINC000018825330,5280961,CHEMBL44,Glycine max and other legumes,31192770;36499286,Genistein,ZINC000018825330,NOT_QUERIED_SKIPPED_AS_REQUESTED,5280961,...,True,3796540,0,Genistein,True,997,997,0,True,True
8,Oleanolic acid,ZINC000003785416,10494,CHEMBL168,"Olea europaea, Ligustrum spp. and many plants",24486850;34798468,Oleanolic acid,ZINC000003785416,NOT_QUERIED_SKIPPED_AS_REQUESTED,10494,...,True,7448934,0,Oleanolic acid,True,988,988,0,True,True
9,Betulinic acid,ZINC000004097714,64971,CHEMBL269277,Betula spp. bark and other plant sources,34269920,Betulinic acid,ZINC000004097714,NOT_QUERIED_SKIPPED_AS_REQUESTED,64971,...,True,7753019,0,Betulinic acid,True,973,973,0,True,True


## 16. Generate a run-specific Word audit report

In [23]:
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CREATE DOCUMENT
# ------------------------------------------------------------

report = Document()

sec = report.sections[0]
sec.top_margin = Inches(0.65)
sec.bottom_margin = Inches(0.65)
sec.left_margin = Inches(0.65)
sec.right_margin = Inches(0.65)

report.styles["Normal"].font.name = "Aptos"
report.styles["Normal"].font.size = Pt(9)

# ------------------------------------------------------------
# 2. TITLE
# ------------------------------------------------------------

p = report.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER

r = p.add_run(
    "ZINC-PRIORITY 31-COMPOUND LIGAND LIBRARY\n"
    "PUBCHEM PARENT RETRIEVAL AND LIGAND PREPARATION REPORT"
)

r.bold = True
r.font.size = Pt(18)
r.font.color.rgb = RGBColor(23, 54, 93)

report.add_paragraph(
    f"Run timestamp (UTC): {datetime.now(timezone.utc).isoformat()}"
)

# ------------------------------------------------------------
# 3. SCOPE
# ------------------------------------------------------------

report.add_heading("Scope", level=1)

report.add_paragraph(
    "This report records PubChem parent structure retrieval by the supplied "
    "CIDs, PubChem 2D similarity retrieval, RDKit chemical QC, ETKDGv3 "
    "conformer generation, MMFF/UFF optimization, SDF writing and read-back "
    "verification with finite, non-degenerate 3D coordinates. Failed API "
    "batches and failed optimisation are not silently accepted. It does not "
    "include receptor preparation or molecular docking."
)

# ------------------------------------------------------------
# 4. PUBCHEM PARENT STRUCTURE RETRIEVAL
# ------------------------------------------------------------

report.add_heading("PubChem parent structure retrieval", level=1)

report.add_paragraph(f"Parents in proposed panel: {len(PANEL)}")
report.add_paragraph(f"PubChem parent structures retrieved: {len(passed)}")
report.add_paragraph(f"Unresolved PubChem parent structures: {len(failed)}")

report.add_paragraph(
    "See PubChem_Parent_CID_Structure_Audit.csv for returned PubChem titles, "
    "SMILES, InChIKeys and lookup outcomes. ZINC IDs are retained as "
    "cross-reference metadata only. CartBlanche/ZINC-22 was skipped."
)

# ------------------------------------------------------------
# 5. PROCESSING PARAMETERS
# ------------------------------------------------------------

report.add_heading("Processing parameters", level=1)

processing_parameters = [
    f"PubChem fast 2D similarity threshold: {SIMILARITY_THRESHOLD}% Tanimoto",
    f"Maximum PubChem records per parent: {MAX_RECORDS_PER_PARENT}",
    "Parent retention: yes; duplicate PubChem CIDs removed in original order",
    "Chemical QC: valid SMILES; at least one carbon; no metals; allowed "
    "elements C/H/N/O/S/P/F/Cl/Br/I; single fragment; no isotope labels",
    f"3D generation: RDKit ETKDGv3, random seed {RANDOM_SEED}, "
    "randomized-coordinate retry",
    "Geometry optimization: MMFF primary; UFF fallback; no optimization "
    "if neither is available",
    "SDF verification: RDKit read-back with removeHs=False; readable "
    "record count and 3D conformer presence reported"
]

for line in processing_parameters:
    report.add_paragraph(line, style="List Bullet")

# ------------------------------------------------------------
# 6. PER-PARENT OUTPUT SUMMARY
# ------------------------------------------------------------

report.add_heading("Per-parent output summary", level=1)

table = report.add_table(rows=1, cols=6)
table.alignment = WD_TABLE_ALIGNMENT.CENTER
table.style = "Light Shading Accent 1"

headers = [
    "Parent",
    "PubChem status",
    "PubChem hits",
    "QC pass",
    "3D success",
    "SDF read-back"
]

for cell, header in zip(table.rows[0].cells, headers):
    cell.text = header

for _, row in summary_df.iterrows():

    cells = table.add_row().cells

    values = [
        row.get("name", ""),
        row.get("PubChem_Status", ""),
        row.get("PubChem_Hits_Returned", ""),
        row.get("QC_Passing", ""),
        row.get("3D_Success", ""),
        row.get("SDF_Readable_Records", "")
    ]

    for cell, value in zip(cells, values):
        cell.text = str(value)

# ------------------------------------------------------------
# 7. INTERPRETATION AND LIMITATIONS
# ------------------------------------------------------------

report.add_heading("Interpretation and limitations", level=1)

report.add_paragraph(
    "ZINC identifiers are retained as cross-reference metadata supplied "
    "for the 31 parent compounds. CartBlanche/ZINC-22 was not queried in "
    "this workflow. Therefore, live ZINC catalogue availability, "
    "purchasability, and exact ZINC-to-PubChem structure correspondence "
    "were not independently verified."
)

report.add_paragraph(
    "PubChem 2D similarity hits are structurally similar compounds, not "
    "proven anticancer agents. The 90% Tanimoto threshold is a chemical "
    "similarity criterion only. Biological activity, target engagement, "
    "selectivity, toxicity, and novelty require separate literature and "
    "experimental evaluation."
)

report.add_paragraph(
    "PubMed results are automated discovery results for follow-up "
    "curation, not a systematic review. ChEMBL identifiers are "
    "cross-checked for resolution. BindingDB similarity-search responses "
    "are preserved, and each returned structure is classified as an "
    "exact parent match or a similar-structure-only record; assay fields "
    "are exported where returned."
)

report.add_paragraph(
    "No target was selected in this workflow. The empty 02_Proteins "
    "directory is intentional. Protein preparation, binding-site "
    "definition, docking, rescoring, molecular dynamics and ADMET are "
    "outside the scope of this notebook."
)

# ------------------------------------------------------------
# 8. SAVE REPORT
# ------------------------------------------------------------

report_path = (
    SUMMARY_ROOT /
    "ZINC_Priority_31_Live_Validation_and_Methods_Report.docx"
)

report.save(report_path)

print("Word report saved:", report_path)
print("Report file exists:", report_path.exists())
print("Parent compounds included:", len(summary_df))

Word report saved: /content/ZINC_Priority_31_Ligand_Library_FINAL/03_Summary/ZINC_Priority_31_Live_Validation_and_Methods_Report.docx
Report file exists: True
Parent compounds included: 31


## 17. Final quantitative checks

In [24]:
# Confirm expected parent-level records
assert len(PANEL) == 31, "Expected 31 parent compounds."
assert len(retrieval_df) == 31, "Retrieval table does not contain 31 parents."
assert len(qc_summary_df) == 31, "QC table does not contain 31 parents."
assert len(sdf_summary_df) == 31, "3D summary does not contain 31 parents."
assert len(verification_df) == 31, "SDF verification does not contain 31 parents."

# Aggregate counts calculated from this run
total_similarity_hits = int(
    retrieval_df["PubChem_Hits_Returned"].fillna(0).sum()
)

total_raw_properties = int(
    qc_summary_df["Raw_property_records"].fillna(0).sum()
)

total_qc_pass = int(
    qc_summary_df["QC_Passing"].fillna(0).sum()
)

total_3d = int(
    sdf_summary_df["3D_Success"].fillna(0).sum()
)

total_sdf_read = int(
    verification_df["SDF_Readable_Records"].fillna(0).sum()
)

total_sdf_unread = int(
    verification_df["Unreadable_Records"].fillna(0).sum()
)

# Final run metrics
run_metrics = pd.DataFrame([{
    "Parent_Count": len(PANEL),
    "PubChem_Parent_Structures_Retrieved": len(passed),
    "PubChem_Parent_Structures_Unresolved": len(failed),
    "ZINC22_Live_Verification_Performed": False,
    "PubChem_Similarity_Hits": total_similarity_hits,
    "PubChem_Property_Records": total_raw_properties,
    "QC_Passing_Records": total_qc_pass,
    "3D_Generated_Records": total_3d,
    "SDF_Readable_Records": total_sdf_read,
    "SDF_Unreadable_Records": total_sdf_unread,
    "All_Readable_SDF_Records_3D": (
        bool(verification_df[
            "All_Readable_Records_Have_Valid_3D_Coordinates"
        ].all())
    ),
    "All_SDF_Readback_Counts_Match": (
        bool(verification_df["Readback_Match"].all())
    )
}])

# Save
metrics_path = SUMMARY_ROOT / "FINAL_RUN_METRICS.csv"
run_metrics.to_csv(metrics_path, index=False)

display(run_metrics)

print("\nPer-parent SDF verification discrepancies:")

discrepancies = verification_df[
    ~verification_df["Readback_Match"]
]

display(discrepancies)

print("\nFinal metrics saved:", metrics_path)
print("SDF discrepancies:", len(discrepancies))

,Parent_Count,PubChem_Parent_Structures_Retrieved,PubChem_Parent_Structures_Unresolved,ZINC22_Live_Verification_Performed,PubChem_Similarity_Hits,PubChem_Property_Records,QC_Passing_Records,3D_Generated_Records,SDF_Readable_Records,SDF_Unreadable_Records,All_Readable_SDF_Records_3D,All_SDF_Readback_Counts_Match
0,31,31,0,False,28124,28124,26066,25896,25896,0,True,True



Per-parent SDF verification discrepancies:


,Parent_Ligand,SDF_Exists,SDF_Generated_Records,SDF_Readable_Records,Unreadable_Records,All_Readable_Records_Have_Valid_3D_Coordinates,Readback_Match



Final metrics saved: /content/ZINC_Priority_31_Ligand_Library_FINAL/03_Summary/FINAL_RUN_METRICS.csv
SDF discrepancies: 0


## 18. Archive the complete project and download the ZIP

In [25]:
from pathlib import Path
import zipfile
from google.colab import files

# Define ZIP output outside the project directory
zip_path = Path(f"/content/{PROJECT_DIR.name}.zip")

# Confirm project directory exists
assert PROJECT_DIR.exists(), "Project directory does not exist."
assert PROJECT_DIR.is_dir(), "Project path is not a directory."

# Remove previous ZIP, if present
if zip_path.exists():
    zip_path.unlink()

# Collect all project files
project_files = sorted(
    path for path in PROJECT_DIR.rglob("*")
    if path.is_file()
)

assert len(project_files) > 0, "No project files found."

# Create ZIP archive
with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in project_files:
        zf.write(
            path,
            arcname=str(path.relative_to(PROJECT_DIR))
        )

# Verify archive integrity
with zipfile.ZipFile(zip_path, "r") as zf:
    bad_file = zf.testzip()
    archived_files = zf.namelist()

assert bad_file is None, f"ZIP integrity check failed: {bad_file}"
assert len(archived_files) == len(project_files), "File count mismatch."

# Final report
print("ZIP created:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / 1024 / 1024, 2))
print("Files archived:", len(archived_files))
print("ZIP integrity: PASS")

print("\nFiles included:")
for path in project_files:
    print(
        f"{path.relative_to(PROJECT_DIR)} "
        f"({path.stat().st_size:,} bytes)"
    )

# Download
files.download(str(zip_path))

ZIP created: /content/ZINC_Priority_31_Ligand_Library_FINAL.zip
ZIP size (MB): 43.92
Files archived: 235
ZIP integrity: PASS

Files included:
01_Ligands/01_Resveratrol/QC_clean/Resveratrol_QC_metadata.csv (346,015 bytes)
01_Ligands/01_Resveratrol/QC_clean/Resveratrol_derivatives_3D_QC_clean.original_backup.sdf (2,256,418 bytes)
01_Ligands/01_Resveratrol/QC_clean/Resveratrol_derivatives_3D_QC_clean.sdf (2,255,702 bytes)
01_Ligands/01_Resveratrol/raw/Resveratrol_QC_exclusions.csv (141,774 bytes)
01_Ligands/01_Resveratrol/raw/Resveratrol_raw_candidates.csv (414,240 bytes)
01_Ligands/01_Resveratrol/raw/Resveratrol_raw_similarity_CIDs.csv (9,424 bytes)
01_Ligands/02_Genistin/QC_clean/Genistin_QC_metadata.csv (906,953 bytes)
01_Ligands/02_Genistin/QC_clean/Genistin_derivatives_3D_QC_clean.original_backup.sdf (5,584,757 bytes)
01_Ligands/02_Genistin/QC_clean/Genistin_derivatives_3D_QC_clean.sdf (5,583,764 bytes)
01_Ligands/02_Genistin/raw/Genistin_QC_exclusions.csv (5,539 bytes)
01_Ligands/02

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
# ============================================================
# STEP 19: CREATE ONE FOLDER CONTAINING ALL 31 SDF FILES
# ============================================================

from pathlib import Path
import shutil
import zipfile
from google.colab import files

PROJECT_DIR = Path("/content/ZINC_Priority_31_Ligand_Library_FINAL")
LIGAND_DIR = PROJECT_DIR / "01_Ligands"

# Destination folder
DOCKING_DIR = PROJECT_DIR / "02_Docking_Input"
DOCKING_DIR.mkdir(parents=True, exist_ok=True)

# Find and copy the 31 individual SDF files
sdf_files = []

for folder in sorted(LIGAND_DIR.iterdir()):

    if not folder.is_dir():
        continue

    matches = list(
        (folder / "QC_clean").glob(
            "*_derivatives_3D_QC_clean.sdf"
        )
    )

    assert len(matches) == 1, (
        f"{folder.name}: expected one SDF, found {len(matches)}"
    )

    sdf_files.append(matches[0])

assert len(sdf_files) == 31, (
    f"Expected 31 SDF files, found {len(sdf_files)}"
)

# Clear previous SDF copies only
for old_file in DOCKING_DIR.glob("*.sdf"):
    old_file.unlink()

for source in sdf_files:
    shutil.copy2(source, DOCKING_DIR / source.name)

# Verify
copied_sdfs = sorted(DOCKING_DIR.glob("*.sdf"))

assert len(copied_sdfs) == 31

for source in sdf_files:
    destination = DOCKING_DIR / source.name
    assert destination.exists()
    assert destination.stat().st_size == source.stat().st_size

print("SDF files copied:", len(copied_sdfs))
print("Destination:", DOCKING_DIR)
print("\nFiles included:")

for i, path in enumerate(copied_sdfs, 1):
    print(f"{i:02d}. {path.name}")

# ------------------------------------------------------------
# CREATE ZIP FOR SUPERVISOR
# ------------------------------------------------------------

zip_path = Path("/content/31_Ligands_For_Docking.zip")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in copied_sdfs:
        zf.write(path, arcname=path.name)

with zipfile.ZipFile(zip_path, "r") as zf:
    assert zf.testzip() is None
    assert len(zf.namelist()) == 31

print("\nZIP created:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / 1024 / 1024, 2))
print("ZIP integrity: PASS")

files.download(str(zip_path))

SDF files copied: 31
Destination: /content/ZINC_Priority_31_Ligand_Library_FINAL/02_Docking_Input

Files included:
01. Andrographolide_derivatives_3D_QC_clean.sdf
02. Apigenin_derivatives_3D_QC_clean.sdf
03. Artemisinin_derivatives_3D_QC_clean.sdf
04. Baicalein_derivatives_3D_QC_clean.sdf
05. Berberine_derivatives_3D_QC_clean.sdf
06. Betulinic acid_derivatives_3D_QC_clean.sdf
07. Caffeic acid_derivatives_3D_QC_clean.sdf
08. Capsaicin_derivatives_3D_QC_clean.sdf
09. Celastrol_derivatives_3D_QC_clean.sdf
10. Diosgenin_derivatives_3D_QC_clean.sdf
11. Ellagic acid_derivatives_3D_QC_clean.sdf
12. Eugenol_derivatives_3D_QC_clean.sdf
13. Genistein_derivatives_3D_QC_clean.sdf
14. Genistin_derivatives_3D_QC_clean.sdf
15. Harmine_derivatives_3D_QC_clean.sdf
16. Honokiol_derivatives_3D_QC_clean.sdf
17. Kaempferol_derivatives_3D_QC_clean.sdf
18. Luteolin_derivatives_3D_QC_clean.sdf
19. Magnolol_derivatives_3D_QC_clean.sdf
20. Myricetin_derivatives_3D_QC_clean.sdf
21. Naringenin_derivatives_3D_QC_c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 19. Final interpretation checklist

Before using the generated structures:

1. Review `PubChem_Parent_CID_Structure_Audit.csv`; confirm each supplied PubChem CID returned a valid structure and inspect compound titles, InChIKeys and SMILES.
2. **ZINC-22 was skipped.** Do not claim that any ZINC ID was live-verified, currently available, purchasable or an exact ZINC/PubChem structure match.
3. Review each `*_QC_exclusions.csv` and `Chemical_QC_Summary.csv`.
4. Confirm that each parent is retained in its own library.
5. Review `SDF_Readback_Verification.csv`; all records must be readable and have finite, non-degenerate 3D coordinates.
6. Review `PubMed_Cancer_Literature_Search.csv` manually and retain only compound-specific, correctly attributed papers.
7. Complete exact-structure BindingDB assay review before claiming experimental ligand–target evidence.
8. Select and justify a protein target separately before docking.

**Do not report the previous 24-parent run's counts (11,616 / 6,544 / 6,528 / 6,527) as results of this new 31-parent run.** The notebook calculates new totals from the current execution. PubChem CID retrieval and ZINC-22 live verification are separate claims; only the former is attempted here.
